**Step 1 — Clean environment setup + imports + seeds + project folders**

In [ ]:
# Step 1: Environment setup (Colab) + imports + reproducibility + repo-like folders

import os, sys, json, random, time, platform
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- Paths ----
ZIP_PATH = Path("/content/drive/MyDrive/archive.zip")  # your provided path
assert ZIP_PATH.exists(), f"ZIP not found at: {ZIP_PATH}"

# Project root inside Colab runtime
PROJECT_ROOT = Path("/content/fl_ids_repo_real")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

# Repo-like structure
for d in [
    "configs",
    "data/raw",
    "data/interim",
    "data/processed",
    "runs",
    "src",
    "src/data",
    "src/models",
    "src/fl",
    "src/utils",
]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# Record environment metadata
env_info = {
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python_version": sys.version,
    "platform": platform.platform(),
    "seed": SEED,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "zip_path": str(ZIP_PATH),
    "project_root": str(PROJECT_ROOT),
}
(PROJECT_ROOT / "runs" / "env_info.json").write_text(json.dumps(env_info, indent=2))

print("✅ Project root:", PROJECT_ROOT)
print("✅ ZIP path:", ZIP_PATH)
print("✅ Saved env info:", PROJECT_ROOT / "runs" / "env_info.json")


**Step 2 — Unzip the dataset into data/raw/ + locate the CSV**

In [ ]:
# Step 2: Unzip archive.zip -> data/raw/ and locate the target CSV

import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/archive.zip")
RAW_DIR = Path("data/raw/edgeiiotset")
RAW_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"ZIP not found: {ZIP_PATH}"

# ---- Extract ----
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    members = z.namelist()
    print(f"✅ ZIP contains {len(members)} entries")
    # Extract all
    z.extractall(RAW_DIR)

print("✅ Extracted to:", RAW_DIR.resolve())

# ---- Find candidate CSV files ----
csv_files = sorted([p for p in RAW_DIR.rglob("*.csv")])

print(f"\nFound {len(csv_files)} CSV file(s). Showing up to 20 paths:")
for p in csv_files[:20]:
    print(" -", p)

# Heuristic: prefer the known Edge-IIoTset DNN CSV name if present
preferred_names = {
    "DNN-EdgeIIoT-dataset.csv",
    "DNN_EdgeIIoT_dataset.csv",
    "DNN-EdgeIIoTset-dataset.csv",
}

target_csv = None
for p in csv_files:
    if p.name in preferred_names:
        target_csv = p
        break

# If not found by exact name, pick the largest CSV (often the main dataset)
if target_csv is None and csv_files:
    target_csv = max(csv_files, key=lambda p: p.stat().st_size)

if target_csv is None:
    raise FileNotFoundError("No CSV files found after extraction. Check the ZIP contents.")

print("\n✅ Selected target CSV:")
print(target_csv.resolve())

# Save selected path for next steps
(Path("configs") / "data_paths.json").write_text(
    json.dumps({"zip_path": str(ZIP_PATH), "raw_dir": str(RAW_DIR), "target_csv": str(target_csv)}, indent=2)
)
print("✅ Saved configs/data_paths.json")


**Step 3 — Load CSV → basic inspection → verify Attack_type + size + missingness**

In [ ]:
# Step 3: Read the selected CSV and inspect the dataset

import json
import pandas as pd
from pathlib import Path

paths = json.loads(Path("configs/data_paths.json").read_text())
CSV_PATH = Path(paths["target_csv"])
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"

# Read (Edge-IIoTset can be large; low_memory=False helps mixed dtypes)
df = pd.read_csv(CSV_PATH, low_memory=False)

print("✅ Loaded:", CSV_PATH.name)
print("Shape:", df.shape)
print("\nColumns (first 30):")
print(list(df.columns)[:30])

# Preview
display(df.head(5))

# Check Attack_type presence
if "Attack_type" in df.columns:
    print("\nAttack_type distribution (top 20):")
    display(df["Attack_type"].value_counts().head(20))
else:
    print("\n⚠️ 'Attack_type' column not found. We'll need to identify the correct label column(s).")

# Missingness quick check
na_total = int(df.isna().sum().sum())
print(f"\nTotal missing cells: {na_total:,}")
print("Top-15 columns by missing values:")
display(df.isna().sum().sort_values(ascending=False).head(15))

# Duplicate rows count (full-row duplicates)
dup_rows = int(df.duplicated().sum())
print(f"\nDuplicate full rows: {dup_rows:,}")


**Step 4 — Cleaning: drop specified columns + drop NaN + drop duplicates + shuffle**

In [ ]:
# Step 4: Cleaning (drop columns, NaN, duplicates, shuffle)

import json
import pandas as pd
from pathlib import Path
from sklearn.utils import shuffle

paths = json.loads(Path("configs/data_paths.json").read_text())
CSV_PATH = Path(paths["target_csv"])
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Loaded:", CSV_PATH.name, "| shape:", df.shape)

drop_columns = [
    "frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
    "http.file_data", "http.request.full_uri", "icmp.transmit_timestamp",
    "http.request.uri.query", "tcp.options", "tcp.payload", "tcp.srcport",
    "tcp.dstport", "udp.port", "mqtt.msg"
]

# Drop only columns that exist
existing = [c for c in drop_columns if c in df.columns]
missing = [c for c in drop_columns if c not in df.columns]

df.drop(existing, axis=1, inplace=True)

print(f"✅ Dropped {len(existing)} columns. Missing (not dropped) {len(missing)} columns.")
if missing:
    print("Missing columns:", missing)

# Drop rows with any NaN
before = df.shape[0]
df.dropna(axis=0, how="any", inplace=True)
print(f"✅ Dropped NaN rows: {before - df.shape[0]:,}")

# Drop duplicates (full-row duplicates)
before = df.shape[0]
df.drop_duplicates(subset=None, keep="first", inplace=True)
print(f"✅ Dropped duplicate rows: {before - df.shape[0]:,}")

# Shuffle
df = shuffle(df, random_state=42).reset_index(drop=True)
print("✅ Shuffled dataset.")

print("\nNew shape:", df.shape)

# Attack_type distribution
if "Attack_type" in df.columns:
    print("\nAttack_type distribution (top 20):")
    display(df["Attack_type"].value_counts().head(20))
else:
    print("\n⚠️ 'Attack_type' not found after cleaning. We must identify the label column.")

# Save cleaned interim file
out_path = Path("data/interim/edgeiiotset_cleaned.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print("\n✅ Saved cleaned CSV to:", out_path.resolve())


**Step 5 — Build labels + one-hot encode categoricals + save encoded arrays for FL**

This will:

* load data/interim/edgeiiotset_cleaned.csv

* create binary Label from Attack_type (Normal → 0, else → 1)

* one-hot encode categorical features (your listed columns if present, otherwise it falls back to all object columns)

* save X, y, attack_type, feature_names into data/processed/edgeiiotset_encoded.npz + metadata JSON

In [ ]:
# Step 5: Label mapping + categorical encoding (one-hot) + save processed arrays

import json
import numpy as np
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path("data/interim/edgeiiotset_cleaned.csv")
assert CLEAN_PATH.exists(), f"Missing cleaned file: {CLEAN_PATH}"

df = pd.read_csv(CLEAN_PATH, low_memory=False)
print("✅ Loaded cleaned CSV:", CLEAN_PATH.name, "| shape:", df.shape)

# ---- 1) Build binary label y from Attack_type ----
assert "Attack_type" in df.columns, "Expected 'Attack_type' column for label mapping."
attack_type = df["Attack_type"].astype(str)

# Map: Normal -> 0, everything else -> 1
y = (attack_type != "Normal").astype(np.int64).to_numpy()

print("Label distribution (0=benign, 1=intrusion):")
print(pd.Series(y).value_counts())

# ---- 2) Prepare feature dataframe (drop labels) ----
feature_df = df.drop(columns=["Attack_type"])

# ---- 3) One-hot encode categoricals ----
# Prefer your specified columns, but only those that exist.
preferred_cat_cols = [
    "http.request.method",
    "http.referer",
    "http.request.version",
    "dns.qry.name.len",
    "mqtt.conack.flags",
    "mqtt.protoname",
    "mqtt.topic",
]

present_preferred = [c for c in preferred_cat_cols if c in feature_df.columns]
object_cols = [c for c in feature_df.columns if feature_df[c].dtype == "object"]

# If any preferred cols exist, encode them + any other object cols (safer for fixed-dim)
# If none exist, encode all object cols.
if present_preferred:
    cat_cols = sorted(set(present_preferred + object_cols))
    print("✅ Encoding preferred categorical cols (plus other object cols).")
else:
    cat_cols = object_cols
    print("✅ Preferred categorical cols not found; encoding all object cols.")

print("Categorical columns to encode:", cat_cols)

# One-hot encode; keep all levels (drop_first=False) to avoid information loss
X_df = pd.get_dummies(feature_df, columns=cat_cols, drop_first=False)

# ---- 4) Ensure numeric + no NaNs ----
if X_df.isna().any().any():
    n_nan = int(X_df.isna().sum().sum())
    raise ValueError(f"Found NaNs in encoded features: total NaNs={n_nan}")

X = X_df.to_numpy(dtype=np.float32)

print("\n✅ Encoded feature matrix ready:")
print("X shape:", X.shape, "| y shape:", y.shape)

# ---- 5) Save processed artifacts ----
PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

npz_path = PROC_DIR / "edgeiiotset_encoded.npz"
meta_path = PROC_DIR / "edgeiiotset_encoded_meta.json"

np.savez_compressed(
    npz_path,
    X=X,
    y=y,
    attack_type=attack_type.to_numpy(),
)

meta = {
    "source_clean_csv": str(CLEAN_PATH),
    "n_rows": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "feature_names": list(X_df.columns),
    "encoded_categorical_columns": cat_cols,
    "label_definition": "Label=1 if Attack_type!='Normal' else 0",
}

meta_path.write_text(json.dumps(meta, indent=2))

print("\n✅ Saved:")
print(" -", npz_path.resolve())
print(" -", meta_path.resolve())


**Step 6 — Dirichlet(𝛼) non-IID partition into K=10 silos + 70/15/15 split within each silo**

This creates splits for 𝛼∈{0.1,0.3,0.5,1.0} and saves them to data/processed/.

In [ ]:
# Step 6: Dirichlet non-IID partition (K=10) + per-silo 70/15/15 split (saved to disk)

import json
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)

K = 10
alpha_grid = [0.1, 0.3, 0.5, 1.0]
split = {"train": 0.70, "val": 0.15, "test": 0.15}

ENC_PATH = Path("data/processed/edgeiiotset_encoded.npz")
META_PATH = Path("data/processed/edgeiiotset_encoded_meta.json")
assert ENC_PATH.exists(), f"Missing encoded dataset: {ENC_PATH}"
assert META_PATH.exists(), f"Missing meta: {META_PATH}"

data = np.load(ENC_PATH, allow_pickle=True)
X = data["X"]
y = data["y"].astype(int)

print("✅ Loaded encoded dataset:", ENC_PATH.name)
print("X shape:", X.shape, "| y shape:", y.shape)

OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def dirichlet_partition_indices(y, K, alpha, seed=42, min_per_silo=50, max_tries=500):
    """
    Label-skew Dirichlet partition with best-effort fallback.
    For each class c: sample proportions p_c ~ Dirichlet(alpha,...,alpha) over K silos,
    then split class indices accordingly.
    """
    rng_local = np.random.default_rng(seed)
    classes = np.unique(y)

    best_silos = None
    best_min_size = -1

    for _ in range(max_tries):
        silos = [[] for _ in range(K)]

        for c in classes:
            idx_c = np.where(y == c)[0]
            rng_local.shuffle(idx_c)

            props = rng_local.dirichlet(alpha=np.full(K, alpha))
            counts = rng_local.multinomial(len(idx_c), props)

            start = 0
            for k in range(K):
                cnt = counts[k]
                if cnt > 0:
                    silos[k].extend(idx_c[start:start+cnt].tolist())
                start += cnt

        sizes = np.array([len(s) for s in silos], dtype=int)
        min_size = int(sizes.min())

        if min_size > best_min_size:
            best_min_size = min_size
            best_silos = [np.array(s, dtype=np.int64) for s in silos]

        if min_size >= min_per_silo:
            for k in range(K):
                rng_local.shuffle(silos[k])
            return [np.array(s, dtype=np.int64) for s in silos], True, best_min_size

    # fallback: return best we saw
    return best_silos, False, best_min_size

def split_silo_indices(idx, y, seed=42, split=(0.70, 0.15, 0.15)):
    """
    Split a silo's indices into train/val/test with stratification when feasible.
    """
    idx = np.array(idx, dtype=np.int64)
    y_s = y[idx]

    # If too small or single-class, do a simple non-stratified split
    if len(idx) < 30 or len(np.unique(y_s)) < 2:
        n = len(idx)
        n_train = max(1, int(split[0]*n))
        n_val   = max(0, int(split[1]*n))
        perm = np.random.default_rng(seed).permutation(idx)
        tr = perm[:n_train]
        va = perm[n_train:n_train+n_val]
        te = perm[n_train+n_val:]
        return tr, va, te

    # train vs temp
    test_val_frac = split[1] + split[2]
    strat_ok = (len(np.unique(y_s)) > 1) and (np.min(np.bincount(y_s)) >= 2)

    tr, temp = train_test_split(
        idx,
        test_size=test_val_frac,
        random_state=seed,
        shuffle=True,
        stratify=y_s if strat_ok else None
    )

    # val vs test
    y_temp = y[temp]
    val_frac_of_temp = split[1] / (split[1] + split[2])
    strat_ok2 = (len(np.unique(y_temp)) > 1) and (np.min(np.bincount(y_temp)) >= 2)

    va, te = train_test_split(
        temp,
        test_size=(1.0 - val_frac_of_temp),
        random_state=seed,
        shuffle=True,
        stratify=y_temp if strat_ok2 else None
    )
    return tr, va, te

# For alpha=0.1 you might get tiny silos; allow smaller min_per_silo
def min_per_silo_for_alpha(a):
    return 50 if a >= 0.5 else 10

all_summaries = {}

for a in alpha_grid:
    min_per = min_per_silo_for_alpha(a)
    silos, met_constraint, best_min = dirichlet_partition_indices(
        y, K=K, alpha=a, seed=SEED, min_per_silo=min_per, max_tries=500
    )

    splits = {"train": [], "val": [], "test": []}
    silo_stats = []

    for k in range(K):
        idx_k = silos[k]
        tr, va, te = split_silo_indices(idx_k, y, seed=SEED, split=(split["train"], split["val"], split["test"]))

        splits["train"].append(tr)
        splits["val"].append(va)
        splits["test"].append(te)

        yk = y[idx_k]
        n0 = int((yk == 0).sum())
        n1 = int((yk == 1).sum())
        silo_stats.append({
            "silo": k,
            "n_total": int(len(idx_k)),
            "n_benign_0": n0,
            "n_intrusion_1": n1,
            "intrusion_rate": float(n1 / max(1, len(idx_k))),
            "n_train": int(len(tr)),
            "n_val": int(len(va)),
            "n_test": int(len(te)),
        })

    out_npz = OUT_DIR / f"splits_edgeiiotset_dirichlet_alpha_{a}.npz"
    np.savez_compressed(
        out_npz,
        train=np.array(splits["train"], dtype=object),
        val=np.array(splits["val"], dtype=object),
        test=np.array(splits["test"], dtype=object),
        alpha=np.array([a], dtype=np.float32),
        K=np.array([K], dtype=np.int32),
        seed=np.array([SEED], dtype=np.int32),
    )

    out_json = OUT_DIR / f"splits_edgeiiotset_dirichlet_alpha_{a}_summary.json"
    summary = {
        "dataset": "Edge-IIoTset (encoded)",
        "alpha": a,
        "K": K,
        "seed": SEED,
        "split_ratios": split,
        "min_per_silo_target": min_per,
        "met_min_per_silo_target": bool(met_constraint),
        "best_min_silo_size_seen": int(best_min),
        "silo_stats": silo_stats,
    }
    out_json.write_text(json.dumps(summary, indent=2))

    sizes = [s["n_total"] for s in silo_stats]
    all_summaries[str(a)] = {
        "npz": str(out_npz),
        "summary": str(out_json),
        "min_silo_size": int(min(sizes)),
        "max_silo_size": int(max(sizes)),
        "met_min_target": bool(met_constraint),
    }

    print(f"✅ alpha={a}: min/max silo size = {min(sizes)}/{max(sizes)} | met_min_target={met_constraint}")

combined_path = OUT_DIR / "splits_edgeiiotset_index.json"
combined_path.write_text(json.dumps(all_summaries, indent=2))
print("✅ Wrote:", combined_path)


**Step 7 — Per-silo local normalization + PyTorch DataLoaders (choose one 𝛼)**

It loads your saved Dirichlet splits, computes (𝜇𝑘,𝜎𝑘) from each silo’s train split only, applies it to that silo’s train/val/test, and saves the norm stats.

In [ ]:
# Step 7: Build per-silo loaders with LOCAL normalization (train-only stats) for a chosen alpha

import json
import numpy as np
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

# ---- Choose alpha here ----
ALPHA = 0.3  # 0.1 / 0.3 / 0.5 / 1.0
BATCH_SIZE = 256
EPS = 1e-8

ENC_PATH = Path("data/processed/edgeiiotset_encoded.npz")
SPL_PATH = Path(f"data/processed/splits_edgeiiotset_dirichlet_alpha_{ALPHA}.npz")
assert ENC_PATH.exists(), f"Missing: {ENC_PATH}"
assert SPL_PATH.exists(), f"Missing: {SPL_PATH}"

data = np.load(ENC_PATH, allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)

spl = np.load(SPL_PATH, allow_pickle=True)
train_idxs = spl["train"]
val_idxs = spl["val"]
test_idxs = spl["test"]
K = int(spl["K"][0])

print("✅ Loaded encoded X,y:", X.shape, y.shape)
print("✅ Loaded splits:", SPL_PATH.name, "| K =", K, "| alpha =", ALPHA)

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

def fit_local_norm(X_train):
    mu = X_train.mean(axis=0, dtype=np.float64)
    sigma = X_train.std(axis=0, dtype=np.float64)
    return mu.astype(np.float32), sigma.astype(np.float32)

def apply_local_norm(X_part, mu, sigma, eps=1e-8):
    return (X_part - mu) / (sigma + eps)

silo_data = []
mu_list, sigma_list = [], []
sizes = []

for k in range(K):
    tr = np.array(train_idxs[k], dtype=np.int64)
    va = np.array(val_idxs[k], dtype=np.int64)
    te = np.array(test_idxs[k], dtype=np.int64)

    X_tr, y_tr = X[tr], y[tr]
    X_va, y_va = X[va], y[va]
    X_te, y_te = X[te], y[te]

    mu_k, sigma_k = fit_local_norm(X_tr)

    X_tr_n = apply_local_norm(X_tr, mu_k, sigma_k, EPS)
    X_va_n = apply_local_norm(X_va, mu_k, sigma_k, EPS)
    X_te_n = apply_local_norm(X_te, mu_k, sigma_k, EPS)

    silo_data.append({
        "train_loader": DataLoader(TabularDataset(X_tr_n, y_tr), batch_size=BATCH_SIZE, shuffle=True),
        "val_loader":   DataLoader(TabularDataset(X_va_n, y_va), batch_size=BATCH_SIZE, shuffle=False),
        "test_loader":  DataLoader(TabularDataset(X_te_n, y_te), batch_size=BATCH_SIZE, shuffle=False),
        "n_train": int(len(tr)),
        "n_val": int(len(va)),
        "n_test": int(len(te)),
    })

    mu_list.append(mu_k)
    sigma_list.append(sigma_k)
    sizes.append((len(tr), len(va), len(te)))

# Save norm stats for reproducibility
out_norm = Path(f"data/processed/local_norm_edgeiiotset_alpha_{ALPHA}.npz")
np.savez_compressed(
    out_norm,
    mu=np.stack(mu_list, axis=0),
    sigma=np.stack(sigma_list, axis=0),
    alpha=np.array([ALPHA], dtype=np.float32),
    K=np.array([K], dtype=np.int32),
    eps=np.array([EPS], dtype=np.float32),
)

print("✅ Saved local norm stats:", out_norm)
print("Per-silo (train,val,test) sizes (first 5):", sizes[:5])
print("Feature dim d =", X.shape[1])

# Keep these in memory for next steps
GLOBAL_D = X.shape[1]
GLOBAL_K = K


**Step 8 — Define the MLP IDS (BN disabled) + weighted BCE + evaluation helpers**

In [ ]:
# Step 8: MLP IDS (FL-stable: no BatchNorm) + weighted BCE + eval utilities (Proposed Methodology MLP Model)

import numpy as np
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", DEVICE)

# ---- Baseline model hyperparams (from your table) ----
HIDDEN = (256, 128, 64)
DROPOUT = 0.30
BATCH_NORM = False   # keep False for FL stability

# Weighted loss (lambda1=2.0, lambda0=1.0) -> pos_weight = 2.0
POS_WEIGHT = torch.tensor([2.0], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)

class MLPIDS(nn.Module):
    """
    Lightweight MLP IDS for tabular flow/session features.
    Outputs logits (sigmoid -> probability).
    """
    def __init__(self, input_dim, hidden=HIDDEN, dropout=DROPOUT, batch_norm=BATCH_NORM):
        super().__init__()
        layers = []
        in_dim = input_dim

        for h in hidden:
            layers.append(nn.Linear(in_dim, h))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h

        self.net = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1)

    def forward(self, x):
        return self.out(self.net(x)).squeeze(-1)  # logits

@torch.no_grad()
def eval_on_loader(model, loader, threshold=0.5):
    model.eval()
    probs_list, y_list = [], []
    total_loss, total_n = 0.0, 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb_f = yb.float().to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb_f)

        bs = xb.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

        probs_list.append(torch.sigmoid(logits).cpu().numpy())
        y_list.append(yb.cpu().numpy())

    probs = np.concatenate(probs_list)
    y_true = np.concatenate(y_list).astype(int)
    y_pred = (probs >= threshold).astype(int)

    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())

    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, (tp+tn+fp+fn))

    return {
        "loss": float(total_loss / max(1,total_n)),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "pred_pos_rate": float(y_pred.mean()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }

@torch.no_grad()
def eval_global_union(model, silo_data, split="val", threshold=0.5):
    # Union-of-silos evaluation (global metrics)
    model.eval()
    probs_all, y_all_list = [], []
    total_loss, total_n = 0.0, 0

    for k in range(len(silo_data)):
        loader = silo_data[k]["val_loader"] if split == "val" else silo_data[k]["test_loader"]
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb_f = yb.float().to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb_f)
            bs = xb.size(0)

            total_loss += float(loss.item()) * bs
            total_n += bs
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
            y_all_list.append(yb.cpu().numpy())

    probs = np.concatenate(probs_all)
    y_true = np.concatenate(y_all_list).astype(int)
    y_pred = (probs >= threshold).astype(int)

    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())

    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, (tp+tn+fp+fn))

    return {
        "loss": float(total_loss / max(1,total_n)),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "pred_pos_rate": float(y_pred.mean()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }

print("✅ Model + evaluation utilities ready.")


In [ ]:
# Step 8A — Logistic Regression IDS (FL-compatible) + weighted BCE + evaluation helpers (Can configure for other Models. Example for Regression)

import numpy as np
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Device:", DEVICE)

# -----------------------------
# Model: Logistic Regression IDS
# -----------------------------
class LogisticIDS(nn.Module):
    """
    Logistic regression for binary intrusion detection.
    Outputs logits (NOT sigmoid).
    """
    def __init__(self, input_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)  # logits


# -----------------------------
# Loss: Weighted BCE (imbalance)
# -----------------------------
POS_WEIGHT = 2.0  # pos_weight ~= lambda1/lambda0 in your weighted CE
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))


# -----------------------------
# Metrics / Evaluation helpers
# -----------------------------
@torch.no_grad()
def eval_loader(model: nn.Module, loader, threshold=0.5):
    model.eval()
    total_loss, n = 0.0, 0
    tp = tn = fp = fn = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.float().to(DEVICE)

        logits = model(xb)
        loss = criterion(logits, yb)

        probs = torch.sigmoid(logits)
        pred = (probs >= threshold).long()
        y = yb.long()

        tp += int(((pred == 1) & (y == 1)).sum().item())
        tn += int(((pred == 0) & (y == 0)).sum().item())
        fp += int(((pred == 1) & (y == 0)).sum().item())
        fn += int(((pred == 0) & (y == 1)).sum().item())

        bsz = xb.size(0)
        total_loss += float(loss.item()) * bsz
        n += bsz

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    acc = (tp + tn) / max(1, (tp + tn + fp + fn))
    fpr = fp / (fp + tn + 1e-12)

    return {
        "loss": total_loss / max(1, n),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "pred_pos_rate": float((tp + fp) / max(1, (tp + tn + fp + fn))),
        "n": int(n),
    }


@torch.no_grad()
def collect_probs(model: nn.Module, loader):
    model.eval()
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs_list.append(torch.sigmoid(logits).cpu().numpy())
        y_list.append(yb.numpy())
    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)


print("✅ Step 8A ready: LogisticIDS + weighted BCE + eval helpers.")


**Step 9 — FedProx local training + update protection + multi-round FedAvg FL on real Edge-IIoTset**

In [ ]:
# Step 9: FL training loop (FedAvg baseline) with FedProx clients + clipped updates

import time, copy, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Assumes Step 7 created: silo_data, GLOBAL_D, GLOBAL_K ----
assert "silo_data" in globals(), "Run Step 7 first (build silo_data)."
d = int(GLOBAL_D)
K = int(GLOBAL_K)

# ---- Training config (baseline-ish; adjust later) ----
CFG = {
    "seed": 42,
    "alpha": 0.3,
    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 1,

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,

    "clip_C": 1.0,      # update clipping
    "dp_sigma": 0.0,    # DP noise disabled; set to 0.4 to enable

    "threshold": 0.5,
}

rng = np.random.default_rng(CFG["seed"])

# ---- Model ----
global_model = MLPIDS(input_dim=d).to(DEVICE)

# ---- Helpers: FedProx training ----
def get_param_vector(model):
    return torch.cat([p.view(-1) for p in model.parameters()])

def fedprox_penalty(model, w0_vec):
    w_vec = get_param_vector(model)
    return 0.5 * torch.sum((w_vec - w0_vec) ** 2)

def train_one_client_fedprox(global_model, train_loader, cfg, seed_offset=0):
    torch.manual_seed(cfg["seed"] + seed_offset)
    model = copy.deepcopy(global_model).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    with torch.no_grad():
        w0 = get_param_vector(model).detach()

    model.train()
    for _ in range(cfg["local_epochs"]):
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.float().to(DEVICE)

            opt.zero_grad()
            logits = model(xb)
            base = criterion(logits, yb)
            prox = cfg["fedprox_mu"] * fedprox_penalty(model, w0)
            loss = base + prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()

    return model

# ---- Helpers: delta, clipping, FedAvg ----
def state_dict_to_vector(sd):
    return torch.cat([v.detach().reshape(-1).float().cpu() for v in sd.values() if torch.is_tensor(v)])

def vector_to_state_dict_like(vec, ref_sd):
    out, off = {}, 0
    for k, v in ref_sd.items():
        if torch.is_tensor(v):
            n = v.numel()
            out[k] = vec[off:off+n].view_as(v).to(v.dtype)
            off += n
        else:
            out[k] = v
    return out

def compute_delta(global_model, local_model):
    g_sd = global_model.state_dict()
    l_sd = local_model.state_dict()
    return state_dict_to_vector(l_sd) - state_dict_to_vector(g_sd), g_sd

def clip_and_noise(delta_vec, C=1.0, sigma=0.0, seed=42):
    delta = delta_vec.clone()
    norm = torch.linalg.norm(delta, ord=2)
    scale = max(1.0, float(norm.item()) / float(C))
    delta = delta / scale
    if sigma and sigma > 0.0:
        gen = torch.Generator(device="cpu").manual_seed(int(seed))
        delta = delta + torch.normal(0.0, float(sigma), size=delta.shape, generator=gen)
    return delta

def payload_bytes(delta_vec): return int(delta_vec.numel() * 4)

def agg_fedavg(deltas, weights):
    W = float(np.sum(weights))
    out = torch.zeros_like(deltas[0])
    for d, w in zip(deltas, weights):
        out += d * (float(w) / W)
    return out

def apply_delta_to_model(global_model, delta_vec, ref_sd, gamma=1.0):
    g_sd = global_model.state_dict()
    delta_sd = vector_to_state_dict_like(delta_vec, ref_sd)
    new_sd = {}
    for k, v in g_sd.items():
        if torch.is_tensor(v):
            new_sd[k] = v + gamma * delta_sd[k].to(v.device)
        else:
            new_sd[k] = v
    global_model.load_state_dict(new_sd)
    return global_model

# ---- Communication sizing ----
param_count = sum(p.numel() for p in global_model.parameters())
model_bytes = int(param_count * 4)
downlink_bytes_per_round = int(K * model_bytes)

# ---- Run directory ----
run_id = time.strftime("run_edgeiiotset_%Y%m%d_%H%M%S", time.gmtime())
RUN_DIR = Path("runs") / run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "config.json").write_text(json.dumps(CFG, indent=2))

rows = []
for t in range(CFG["rounds"]):
    # participation with dropout
    participants = [k for k in range(K) if rng.uniform() > CFG["dropout_q"]]
    if len(participants) < CFG["participation_min"]:
        participants = list(rng.choice(K, size=CFG["participation_min"], replace=False))

    pre_val = eval_global_union(global_model, silo_data, split="val", threshold=CFG["threshold"])

    t0 = time.time()
    deltas, weights = [], []
    uplink_bytes = 0
    ref_sd = None

    for k in participants:
        local_model = train_one_client_fedprox(global_model, silo_data[k]["train_loader"], CFG, seed_offset=1000*t + k)
        delta_vec, ref_sd = compute_delta(global_model, local_model)
        protected = clip_and_noise(delta_vec, C=CFG["clip_C"], sigma=CFG["dp_sigma"], seed=CFG["seed"] + 1000*t + k)
        deltas.append(protected)
        weights.append(silo_data[k]["n_train"])
        uplink_bytes += payload_bytes(protected)

    agg_delta = agg_fedavg(deltas, weights)
    global_model = apply_delta_to_model(global_model, agg_delta, ref_sd, gamma=CFG["server_gamma"])

    round_sec = time.time() - t0
    post_val = eval_global_union(global_model, silo_data, split="val", threshold=CFG["threshold"])

    rows.append({
        "round": t,
        "participants": len(participants),
        "pre_val_f1": pre_val["f1"],
        "post_val_f1": post_val["f1"],
        "pre_pred_pos_rate": pre_val["pred_pos_rate"],
        "post_pred_pos_rate": post_val["pred_pos_rate"],
        "uplink_MB": uplink_bytes / 1e6,
        "downlink_MB": downlink_bytes_per_round / 1e6,
        "round_seconds": round_sec,
    })

    print(f"Round {t}: m={len(participants)} | val_f1 {pre_val['f1']:.4f}->{post_val['f1']:.4f} | "
          f"pred+ {pre_val['pred_pos_rate']:.3f}->{post_val['pred_pos_rate']:.3f} | "
          f"up={uplink_bytes/1e6:.3f}MB down={downlink_bytes_per_round/1e6:.3f}MB sec={round_sec:.2f}")

# Final test (global union)
final_test = eval_global_union(global_model, silo_data, split="test", threshold=CFG["threshold"])

# Save artifacts
import pandas as pd
pd.DataFrame(rows).to_csv(RUN_DIR / "round_log.csv", index=False)
(RUN_DIR / "final_test.json").write_text(json.dumps(final_test, indent=2))
torch.save(global_model.state_dict(), RUN_DIR / "global_model_final.pt")

print("\n✅ Saved run artifacts to:", RUN_DIR)
print(" - round_log.csv")
print(" - final_test.json")
print(" - global_model_final.pt")
print("\n📌 Final global TEST:", final_test)


**Step 10 — Per-silo threshold calibration 𝜃𝑘 on validation + per-silo & global test metrics**

In [ ]:
# Step 10: Calibrate theta_k per silo using VAL, then evaluate per-silo & global TEST

import json
import numpy as np
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Use the most recent run from Step 9 (or set manually) ----
# If you want to set manually: RUN_DIR = Path("runs/<your_run_id>")
runs = sorted(Path("runs").glob("run_edgeiiotset_*"))
assert len(runs) > 0, "No run_edgeiiotset_* found. Run Step 9 first."
RUN_DIR = runs[-1]
print("✅ Using RUN_DIR:", RUN_DIR)

cfg = json.loads((RUN_DIR / "config.json").read_text())
threshold_default = float(cfg.get("threshold", 0.5))

# ---- Calibration mode ----
CALIBRATION_MODE = "fpr"     # "fpr" or "f1"
TARGET_FPR = 0.05            # used only if CALIBRATION_MODE == "fpr"

assert "silo_data" in globals(), "Run Step 7 first (build silo_data)."
d = int(GLOBAL_D)
K = int(GLOBAL_K)

# ---- Load model weights from Step 9 ----
model = MLPIDS(input_dim=d).to(DEVICE)
model.load_state_dict(torch.load(RUN_DIR / "global_model_final.pt", map_location=DEVICE))
model.eval()

@torch.no_grad()
def collect_probs(loader):
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs_list.append(torch.sigmoid(logits).cpu().numpy())
        y_list.append(yb.cpu().numpy())
    probs = np.concatenate(probs_list).astype(np.float64)
    y_true = np.concatenate(y_list).astype(int)
    return probs, y_true

def metrics_at_threshold(probs, y_true, thr):
    y_pred = (probs >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())

    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, (tp+tn+fp+fn))
    fpr       = fp / (fp + tn + 1e-12)

    return {
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "pred_pos_rate": float(y_pred.mean()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }

def calibrate_theta_fpr(probs_val, y_val, target_fpr=0.05):
    benign_probs = probs_val[y_val == 0]
    if benign_probs.size == 0:
        return threshold_default
    # Threshold at (1-target_fpr) quantile of benign probs
    return float(np.quantile(benign_probs, 1.0 - target_fpr))

def calibrate_theta_max_f1(probs_val, y_val):
    grid = np.linspace(0.01, 0.99, 99)
    best_thr, best_f1 = threshold_default, -1.0
    for thr in grid:
        m = metrics_at_threshold(probs_val, y_val, thr)
        if m["f1"] > best_f1:
            best_f1 = m["f1"]
            best_thr = float(thr)
    return best_thr

# ---- Per-silo calibration + evaluation ----
thetas = []
per_silo_val = []
per_silo_test = []

for k in range(K):
    probs_val, y_val = collect_probs(silo_data[k]["val_loader"])
    if CALIBRATION_MODE == "fpr":
        theta_k = calibrate_theta_fpr(probs_val, y_val, target_fpr=TARGET_FPR)
    else:
        theta_k = calibrate_theta_max_f1(probs_val, y_val)

    thetas.append(theta_k)
    per_silo_val.append(metrics_at_threshold(probs_val, y_val, theta_k))

    probs_test, y_test = collect_probs(silo_data[k]["test_loader"])
    per_silo_test.append(metrics_at_threshold(probs_test, y_test, theta_k))

# ---- Global union metrics (sum confusion across silos) ----
tp = sum(m["tp"] for m in per_silo_test)
tn = sum(m["tn"] for m in per_silo_test)
fp = sum(m["fp"] for m in per_silo_test)
fn = sum(m["fn"] for m in per_silo_test)

precision = tp / (tp + fp + 1e-12)
recall    = tp / (tp + fn + 1e-12)
f1        = 2*precision*recall / (precision + recall + 1e-12)
acc       = (tp + tn) / max(1, tp+tn+fp+fn)
fpr       = fp / (fp + tn + 1e-12)

global_metrics = {
    "acc": float(acc),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "fpr": float(fpr),
    "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
}

print(f"✅ Calibration mode: {CALIBRATION_MODE} | target_fpr={TARGET_FPR if CALIBRATION_MODE=='fpr' else 'N/A'}")
print("Theta_k (first 5):", [round(t, 4) for t in thetas[:5]])
print("\n📌 Global TEST with per-silo theta_k:", global_metrics)

# ---- Per-silo table ----
import pandas as pd
df = pd.DataFrame([{
    "silo": k,
    "theta_k": thetas[k],
    "val_f1": per_silo_val[k]["f1"],
    "test_f1": per_silo_test[k]["f1"],
    "test_precision": per_silo_test[k]["precision"],
    "test_recall": per_silo_test[k]["recall"],
    "test_fpr": per_silo_test[k]["fpr"],
    "pred_pos_rate": per_silo_test[k]["pred_pos_rate"],
    "n_test": silo_data[k]["n_test"],
} for k in range(K)])
display(df)

# ---- Save artifacts ----
out = RUN_DIR / f"theta_calibration_{CALIBRATION_MODE}.json"
out.write_text(json.dumps({
    "calibration_mode": CALIBRATION_MODE,
    "target_fpr": TARGET_FPR,
    "thetas": thetas,
    "global_test": global_metrics,
    "per_silo_val": per_silo_val,
    "per_silo_test": per_silo_test,
}, indent=2))
print("✅ Saved:", out)


**Next step (Step 10.1) — Fix calibration: enforce sane 𝜃𝑘 and guardrails**

In [ ]:
# Step 10.1: Robust per-silo theta calibration with bounds + fallbacks + sanity checks

import json
import numpy as np
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use latest run from Step 9
runs = sorted(Path("runs").glob("run_edgeiiotset_*"))
assert len(runs) > 0, "No run_edgeiiotset_* found. Run Step 9 first."
RUN_DIR = runs[-1]
print("✅ Using RUN_DIR:", RUN_DIR)

cfg = json.loads((RUN_DIR / "config.json").read_text())
default_theta = float(cfg.get("threshold", 0.5))

assert "silo_data" in globals(), "Run Step 7 first (build silo_data)."
d = int(GLOBAL_D); K = int(GLOBAL_K)

# Load trained model
model = MLPIDS(input_dim=d).to(DEVICE)
model.load_state_dict(torch.load(RUN_DIR / "global_model_final.pt", map_location=DEVICE))
model.eval()

# Calibration settings
TARGET_FPR = 0.05
THETA_MIN = 0.05
THETA_MAX = 0.95
MIN_BENIGN_VAL = 200          # if fewer benign val samples, fallback
MAX_PRED_POS = 0.90           # sanity upper bound
MIN_PRED_POS = 0.01           # sanity lower bound

@torch.no_grad()
def collect_probs(loader):
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        probs_list.append(torch.sigmoid(model(xb)).cpu().numpy())
        y_list.append(yb.cpu().numpy())
    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)

def metrics_at_threshold(probs, y_true, thr):
    y_pred = (probs >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)
    return {
        "acc": float(acc), "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "fpr": float(fpr), "pred_pos_rate": float(y_pred.mean()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }

def robust_theta_from_fpr(probs_val, y_val, target_fpr):
    benign = probs_val[y_val == 0]
    if benign.size < MIN_BENIGN_VAL:
        return default_theta, f"fallback_default (benign_val={benign.size})"
    thr = float(np.quantile(benign, 1.0 - target_fpr))
    thr = float(np.clip(thr, THETA_MIN, THETA_MAX))
    return thr, "quantile_benign"

# Calibrate per silo with sanity checks
thetas = []
notes = []
per_silo_test = []

for k in range(K):
    probs_val, y_val = collect_probs(silo_data[k]["val_loader"])
    theta_k, note = robust_theta_from_fpr(probs_val, y_val, TARGET_FPR)

    # sanity check on val predicted positive rate
    m_val = metrics_at_threshold(probs_val, y_val, theta_k)
    if m_val["pred_pos_rate"] > MAX_PRED_POS:
        theta_k = min(THETA_MAX, max(theta_k, 0.80))
        note += " | adjusted_high_pred_pos"
    elif m_val["pred_pos_rate"] < MIN_PRED_POS:
        theta_k = max(THETA_MIN, min(theta_k, 0.20))
        note += " | adjusted_low_pred_pos"

    thetas.append(theta_k)
    notes.append(note)

    probs_test, y_test = collect_probs(silo_data[k]["test_loader"])
    per_silo_test.append(metrics_at_threshold(probs_test, y_test, theta_k))

# Global union
tp = sum(m["tp"] for m in per_silo_test)
tn = sum(m["tn"] for m in per_silo_test)
fp = sum(m["fp"] for m in per_silo_test)
fn = sum(m["fn"] for m in per_silo_test)
precision = tp / (tp + fp + 1e-12)
recall    = tp / (tp + fn + 1e-12)
f1        = 2*precision*recall / (precision + recall + 1e-12)
acc       = (tp + tn) / max(1, tp+tn+fp+fn)
fpr       = fp / (fp + tn + 1e-12)

global_metrics = {
    "acc": float(acc), "precision": float(precision), "recall": float(recall),
    "f1": float(f1), "fpr": float(fpr),
    "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
}

print("\n📌 Global TEST with robust per-silo theta_k:", global_metrics)

# Per-silo summary table
import pandas as pd
df = pd.DataFrame([{
    "silo": k,
    "theta_k": thetas[k],
    "note": notes[k],
    "test_f1": per_silo_test[k]["f1"],
    "test_precision": per_silo_test[k]["precision"],
    "test_recall": per_silo_test[k]["recall"],
    "test_fpr": per_silo_test[k]["fpr"],
    "pred_pos_rate": per_silo_test[k]["pred_pos_rate"],
    "n_test": silo_data[k]["n_test"],
} for k in range(K)])
display(df)

# Save artifact
out = RUN_DIR / "theta_calibration_fpr_robust.json"
out.write_text(json.dumps({
    "target_fpr": TARGET_FPR,
    "theta_min": THETA_MIN,
    "theta_max": THETA_MAX,
    "min_benign_val": MIN_BENIGN_VAL,
    "thetas": thetas,
    "notes": notes,
    "global_test": global_metrics,
    "per_silo_test": per_silo_test,
}, indent=2))
print("✅ Saved:", out)


**Next step — Step 11: FedAvg vs TrimMean𝛽 on real Edge-IIoTset + robust 𝜃𝑘 evaluation**

In [ ]:
# Step 11: Controlled FedAvg vs TrimMean(beta) comparison on REAL Edge-IIoTset
#          + robust per-silo theta_k evaluation (FPR-target with guardrails)

import time, copy, json
import numpy as np
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert "silo_data" in globals(), "Run Step 7 first."
d = int(GLOBAL_D); K = int(GLOBAL_K)

# -----------------------------
# Shared config for both runs
# -----------------------------
BASE = {
    "seed": 42,
    "alpha": 0.3,
    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 6,   # IMPORTANT: ensure TrimMean has enough participants

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,

    "clip_C": 1.0,
    "dp_sigma": 0.0,

    "threshold": 0.5,
}

TRIM_BETA = 0.10  # you can tune later

# -----------------------------
# Local training (FedProx)
# -----------------------------
def get_param_vector(model):
    return torch.cat([p.view(-1) for p in model.parameters()])

def fedprox_penalty(model, w0_vec):
    w_vec = get_param_vector(model)
    return 0.5 * torch.sum((w_vec - w0_vec) ** 2)

def train_one_client_fedprox(global_model, train_loader, cfg, seed_offset=0):
    torch.manual_seed(cfg["seed"] + seed_offset)
    model = copy.deepcopy(global_model).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    with torch.no_grad():
        w0 = get_param_vector(model).detach()

    model.train()
    for _ in range(cfg["local_epochs"]):
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.float().to(DEVICE)

            opt.zero_grad()
            logits = model(xb)
            base = criterion(logits, yb)
            prox = cfg["fedprox_mu"] * fedprox_penalty(model, w0)
            loss = base + prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
    return model

# -----------------------------
# Delta/protection/aggregation
# -----------------------------
def state_dict_to_vector(sd):
    return torch.cat([v.detach().reshape(-1).float().cpu() for v in sd.values() if torch.is_tensor(v)])

def vector_to_state_dict_like(vec, ref_sd):
    out, off = {}, 0
    for k, v in ref_sd.items():
        if torch.is_tensor(v):
            n = v.numel()
            out[k] = vec[off:off+n].view_as(v).to(v.dtype)
            off += n
        else:
            out[k] = v
    return out

def compute_delta(global_model, local_model):
    g_sd = global_model.state_dict()
    l_sd = local_model.state_dict()
    return state_dict_to_vector(l_sd) - state_dict_to_vector(g_sd), g_sd

def clip_and_noise(delta_vec, C=1.0, sigma=0.0, seed=42):
    delta = delta_vec.clone()
    norm = torch.linalg.norm(delta, ord=2)
    scale = max(1.0, float(norm.item()) / float(C))
    delta = delta / scale
    if sigma and sigma > 0.0:
        gen = torch.Generator(device="cpu").manual_seed(int(seed))
        delta = delta + torch.normal(0.0, float(sigma), size=delta.shape, generator=gen)
    return delta

def agg_fedavg(deltas, weights):
    W = float(np.sum(weights))
    out = torch.zeros_like(deltas[0])
    for d, w in zip(deltas, weights):
        out += d * (float(w) / W)
    return out

def agg_trimmed_mean(deltas, beta=0.10):
    m = len(deltas)
    X = torch.stack(deltas, dim=0)  # [m,P]
    trim = int(np.floor(beta * m))
    if 2 * trim >= m:
        # Not enough participants to trim safely
        return torch.mean(X, dim=0)
    Xs, _ = torch.sort(X, dim=0)
    Xk = Xs[trim:m-trim, :]
    return torch.mean(Xk, dim=0)

def apply_delta_to_model(global_model, delta_vec, ref_sd, gamma=1.0):
    g_sd = global_model.state_dict()
    delta_sd = vector_to_state_dict_like(delta_vec, ref_sd)
    new_sd = {}
    for k, v in g_sd.items():
        if torch.is_tensor(v):
            new_sd[k] = v + gamma * delta_sd[k].to(v.device)
        else:
            new_sd[k] = v
    global_model.load_state_dict(new_sd)
    return global_model

# -----------------------------
# Robust theta evaluation (same logic as Step 10.1)
# -----------------------------
TARGET_FPR = 0.05
THETA_MIN = 0.05
THETA_MAX = 0.95
MIN_BENIGN_VAL = 200
MAX_PRED_POS = 0.90
MIN_PRED_POS = 0.01

@torch.no_grad()
def collect_probs(model, loader):
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        probs_list.append(torch.sigmoid(model(xb)).cpu().numpy())
        y_list.append(yb.cpu().numpy())
    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)

def metrics_at_threshold(probs, y_true, thr):
    y_pred = (probs >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)
    return {"acc":acc,"precision":precision,"recall":recall,"f1":f1,"fpr":fpr,
            "tp":tp,"tn":tn,"fp":fp,"fn":fn,"pred_pos_rate":float(y_pred.mean())}

def robust_theta_from_fpr(probs_val, y_val, default_theta):
    benign = probs_val[y_val == 0]
    if benign.size < MIN_BENIGN_VAL:
        return default_theta, f"fallback_default (benign_val={benign.size})"
    thr = float(np.quantile(benign, 1.0 - TARGET_FPR))
    thr = float(np.clip(thr, THETA_MIN, THETA_MAX))
    return thr, "quantile_benign"

def eval_with_robust_thetas(model, default_theta):
    thetas, notes, per_silo = [], [], []
    for k in range(K):
        probs_val, y_val = collect_probs(model, silo_data[k]["val_loader"])
        theta_k, note = robust_theta_from_fpr(probs_val, y_val, default_theta)
        m_val = metrics_at_threshold(probs_val, y_val, theta_k)
        if m_val["pred_pos_rate"] > MAX_PRED_POS:
            theta_k = min(THETA_MAX, max(theta_k, 0.80))
            note += " | adjusted_high_pred_pos"
        elif m_val["pred_pos_rate"] < MIN_PRED_POS:
            theta_k = max(THETA_MIN, min(theta_k, 0.20))
            note += " | adjusted_low_pred_pos"

        probs_test, y_test = collect_probs(model, silo_data[k]["test_loader"])
        m_test = metrics_at_threshold(probs_test, y_test, theta_k)

        thetas.append(theta_k); notes.append(note); per_silo.append(m_test)

    tp = sum(m["tp"] for m in per_silo)
    tn = sum(m["tn"] for m in per_silo)
    fp = sum(m["fp"] for m in per_silo)
    fn = sum(m["fn"] for m in per_silo)
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)

    global_m = {"acc":float(acc), "precision":float(precision), "recall":float(recall),
                "f1":float(f1), "fpr":float(fpr), "tp":int(tp), "tn":int(tn), "fp":int(fp), "fn":int(fn)}
    return global_m, thetas, notes

# -----------------------------
# Simulator (runs one aggregator)
# -----------------------------
def run_fl(aggregator="fedavg"):
    cfg = dict(BASE)
    cfg["aggregator"] = aggregator
    cfg["trim_beta"] = TRIM_BETA

    rng = np.random.default_rng(cfg["seed"])
    model = MLPIDS(input_dim=d).to(DEVICE)

    run_id = time.strftime(f"run_real_cmp_{aggregator}_%Y%m%d_%H%M%S", time.gmtime())
    RUN_DIR = Path("runs") / run_id
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    (RUN_DIR / "config.json").write_text(json.dumps(cfg, indent=2))

    for t in range(cfg["rounds"]):
        participants = [k for k in range(K) if rng.uniform() > cfg["dropout_q"]]
        if len(participants) < cfg["participation_min"]:
            participants = list(rng.choice(K, size=cfg["participation_min"], replace=False))

        deltas, weights = [], []
        ref_sd = None
        for k in participants:
            local_model = train_one_client_fedprox(model, silo_data[k]["train_loader"], cfg, seed_offset=1000*t + k)
            delta_vec, ref_sd = compute_delta(model, local_model)
            protected = clip_and_noise(delta_vec, C=cfg["clip_C"], sigma=cfg["dp_sigma"], seed=cfg["seed"] + 1000*t + k)
            deltas.append(protected)
            weights.append(silo_data[k]["n_train"])

        if aggregator == "fedavg":
            agg_delta = agg_fedavg(deltas, weights)
        else:
            agg_delta = agg_trimmed_mean(deltas, beta=cfg["trim_beta"])

        model = apply_delta_to_model(model, agg_delta, ref_sd, gamma=cfg["server_gamma"])
        print(f"{aggregator} | round {t} done | participants={len(participants)}")

    # Evaluate with robust theta_k
    global_m, thetas, notes = eval_with_robust_thetas(model, default_theta=cfg["threshold"])

    (RUN_DIR / "global_test_robust_theta.json").write_text(json.dumps(global_m, indent=2))
    torch.save(model.state_dict(), RUN_DIR / "global_model_final.pt")

    return RUN_DIR, global_m

# -----------------------------
# Run both and compare
# -----------------------------
fed_dir, fed_m = run_fl("fedavg")
trim_dir, trim_m = run_fl("trimmean")

print("\n✅ Done.")
print("FedAvg run   :", fed_dir)
print("TrimMean run :", trim_dir)
print("\nFinal GLOBAL TEST (robust theta_k):")
print("FedAvg  :", fed_m)
print("TrimMean:", trim_m)


**Next step — Step 12: Add FL-stage adversary simulation (poisoning / backdoor-ready scaffold)**

In [ ]:
# Step 12: Adversarial FL simulation (poisoning via sign-flip) + compare FedAvg vs TrimMean

import time, copy, json
import numpy as np
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert "silo_data" in globals(), "Run Step 7 first."
d = int(GLOBAL_D); K = int(GLOBAL_K)

# -----------------------------
# Config
# -----------------------------
CFG = {
    "seed": 42,
    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 6,   # keep >=6 so TrimMean has enough participants

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,
    "clip_C": 1.0,
    "dp_sigma": 0.0,

    "threshold": 0.5,

    # adversary
    "malicious_frac": 0.2,    # alpha in your threat model (e.g., 0.1, 0.2, 0.3)
    "poison_scale": 5.0,      # scale for flipped update (try 1.0, 5.0, 10.0)
}

TRIM_BETA = 0.10

# -----------------------------
# Local training (FedProx)
# -----------------------------
def get_param_vector(model):
    return torch.cat([p.view(-1) for p in model.parameters()])

def fedprox_penalty(model, w0_vec):
    w_vec = get_param_vector(model)
    return 0.5 * torch.sum((w_vec - w0_vec) ** 2)

def train_one_client_fedprox(global_model, train_loader, cfg, seed_offset=0):
    torch.manual_seed(cfg["seed"] + seed_offset)
    model = copy.deepcopy(global_model).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    with torch.no_grad():
        w0 = get_param_vector(model).detach()

    model.train()
    for _ in range(cfg["local_epochs"]):
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.float().to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            base = criterion(logits, yb)
            prox = cfg["fedprox_mu"] * fedprox_penalty(model, w0)
            loss = base + prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
    return model

# -----------------------------
# Delta/protection/aggregation
# -----------------------------
def state_dict_to_vector(sd):
    return torch.cat([v.detach().reshape(-1).float().cpu() for v in sd.values() if torch.is_tensor(v)])

def vector_to_state_dict_like(vec, ref_sd):
    out, off = {}, 0
    for k, v in ref_sd.items():
        if torch.is_tensor(v):
            n = v.numel()
            out[k] = vec[off:off+n].view_as(v).to(v.dtype)
            off += n
        else:
            out[k] = v
    return out

def compute_delta(global_model, local_model):
    g_sd = global_model.state_dict()
    l_sd = local_model.state_dict()
    return state_dict_to_vector(l_sd) - state_dict_to_vector(g_sd), g_sd

def clip_and_noise(delta_vec, C=1.0, sigma=0.0, seed=42):
    delta = delta_vec.clone()
    norm = torch.linalg.norm(delta, ord=2)
    scale = max(1.0, float(norm.item()) / float(C))
    delta = delta / scale
    if sigma and sigma > 0.0:
        gen = torch.Generator(device="cpu").manual_seed(int(seed))
        delta = delta + torch.normal(0.0, float(sigma), size=delta.shape, generator=gen)
    return delta

def agg_fedavg(deltas, weights):
    W = float(np.sum(weights))
    out = torch.zeros_like(deltas[0])
    for dlt, w in zip(deltas, weights):
        out += dlt * (float(w) / W)
    return out

def agg_trimmed_mean(deltas, beta=0.10):
    m = len(deltas)
    X = torch.stack(deltas, dim=0)
    trim = int(np.floor(beta * m))
    if 2 * trim >= m:
        return torch.mean(X, dim=0)
    Xs, _ = torch.sort(X, dim=0)
    Xk = Xs[trim:m-trim, :]
    return torch.mean(Xk, dim=0)

def apply_delta_to_model(global_model, delta_vec, ref_sd, gamma=1.0):
    g_sd = global_model.state_dict()
    delta_sd = vector_to_state_dict_like(delta_vec, ref_sd)
    new_sd = {}
    for k, v in g_sd.items():
        if torch.is_tensor(v):
            new_sd[k] = v + gamma * delta_sd[k].to(v.device)
        else:
            new_sd[k] = v
    global_model.load_state_dict(new_sd)
    return global_model

# -----------------------------
# Robust theta eval (reuse from Step 11)
# -----------------------------
TARGET_FPR = 0.05
THETA_MIN = 0.05
THETA_MAX = 0.95
MIN_BENIGN_VAL = 200
MAX_PRED_POS = 0.90
MIN_PRED_POS = 0.01

@torch.no_grad()
def collect_probs(model, loader):
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        probs_list.append(torch.sigmoid(model(xb)).cpu().numpy())
        y_list.append(yb.cpu().numpy())
    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)

def metrics_at_threshold(probs, y_true, thr):
    y_pred = (probs >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)
    return {"acc":float(acc),"precision":float(precision),"recall":float(recall),"f1":float(f1),"fpr":float(fpr),
            "tp":tp,"tn":tn,"fp":fp,"fn":fn,"pred_pos_rate":float(y_pred.mean())}

def robust_theta_from_fpr(probs_val, y_val, default_theta):
    benign = probs_val[y_val == 0]
    if benign.size < MIN_BENIGN_VAL:
        return default_theta, f"fallback_default (benign_val={benign.size})"
    thr = float(np.quantile(benign, 1.0 - TARGET_FPR))
    thr = float(np.clip(thr, THETA_MIN, THETA_MAX))
    return thr, "quantile_benign"

def eval_with_robust_thetas(model, default_theta):
    per_silo = []
    for k in range(K):
        probs_val, y_val = collect_probs(model, silo_data[k]["val_loader"])
        theta_k, note = robust_theta_from_fpr(probs_val, y_val, default_theta)
        m_val = metrics_at_threshold(probs_val, y_val, theta_k)
        if m_val["pred_pos_rate"] > MAX_PRED_POS:
            theta_k = min(THETA_MAX, max(theta_k, 0.80))
        elif m_val["pred_pos_rate"] < MIN_PRED_POS:
            theta_k = max(THETA_MIN, min(theta_k, 0.20))

        probs_test, y_test = collect_probs(model, silo_data[k]["test_loader"])
        per_silo.append(metrics_at_threshold(probs_test, y_test, theta_k))

    tp = sum(m["tp"] for m in per_silo); tn = sum(m["tn"] for m in per_silo)
    fp = sum(m["fp"] for m in per_silo); fn = sum(m["fn"] for m in per_silo)
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)
    return {"acc":acc,"precision":precision,"recall":recall,"f1":f1,"fpr":fpr,"tp":tp,"tn":tn,"fp":fp,"fn":fn}

# -----------------------------
# Run FL with attack
# -----------------------------
def run_fl(aggregator="fedavg"):
    cfg = dict(CFG)
    cfg["aggregator"] = aggregator
    cfg["trim_beta"] = TRIM_BETA

    rng = np.random.default_rng(cfg["seed"])
    model = MLPIDS(input_dim=d).to(DEVICE)

    for t in range(cfg["rounds"]):
        participants = [k for k in range(K) if rng.uniform() > cfg["dropout_q"]]
        if len(participants) < cfg["participation_min"]:
            participants = list(rng.choice(K, size=cfg["participation_min"], replace=False))

        # choose malicious subset among participants
        m_count = int(np.floor(cfg["malicious_frac"] * len(participants)))
        malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()

        deltas, weights = [], []
        ref_sd = None

        for k in participants:
            local_model = train_one_client_fedprox(model, silo_data[k]["train_loader"], cfg, seed_offset=1000*t + k)
            delta_vec, ref_sd = compute_delta(model, local_model)
            protected = clip_and_noise(delta_vec, C=cfg["clip_C"], sigma=cfg["dp_sigma"], seed=cfg["seed"] + 1000*t + k)

            # poisoning: sign-flip + scale
            if k in malicious:
                protected = -cfg["poison_scale"] * protected

            deltas.append(protected)
            weights.append(silo_data[k]["n_train"])

        if aggregator == "fedavg":
            agg_delta = agg_fedavg(deltas, weights)
        else:
            agg_delta = agg_trimmed_mean(deltas, beta=cfg["trim_beta"])

        model = apply_delta_to_model(model, agg_delta, ref_sd, gamma=cfg["server_gamma"])
        print(f"{aggregator} | round {t} done | participants={len(participants)} | malicious={len(malicious)}")

    return eval_with_robust_thetas(model, default_theta=cfg["threshold"])

fed_m = run_fl("fedavg")
trim_m = run_fl("trimmean")

print("\n✅ Final GLOBAL TEST under poisoning (robust theta_k):")
print("FedAvg  :", {k: float(v) if isinstance(v, (np.floating,float)) else int(v) for k,v in fed_m.items()})
print("TrimMean:", {k: float(v) if isinstance(v, (np.floating,float)) else int(v) for k,v in trim_m.items()})


**Next step — Step 13: Robustness sweep over malicious fraction
𝛼 and poison strength**

In [ ]:
# Step 13: Robustness sweep (malicious_frac x poison_scale) for FedAvg vs TrimMean

import copy
import numpy as np
import pandas as pd
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert "silo_data" in globals(), "Run Step 7 first."
d = int(GLOBAL_D); K = int(GLOBAL_K)

# Base training config (keep same as Step 12 for apples-to-apples)
BASE = {
    "seed": 42,
    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 6,

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,
    "clip_C": 1.0,
    "dp_sigma": 0.0,

    "threshold": 0.5,
}
TRIM_BETA = 0.10

# Reuse helpers from Step 12 if present; otherwise fail loudly
needed = [
    "train_one_client_fedprox", "compute_delta", "clip_and_noise",
    "agg_fedavg", "agg_trimmed_mean", "apply_delta_to_model",
    "eval_with_robust_thetas"
]
for fn in needed:
    assert fn in globals(), f"Missing helper '{fn}'. Re-run Step 12 cell first."

def run_poisoned(aggregator, malicious_frac, poison_scale):
    cfg = dict(BASE)
    cfg["malicious_frac"] = float(malicious_frac)
    cfg["poison_scale"] = float(poison_scale)

    rng = np.random.default_rng(cfg["seed"])
    model = MLPIDS(input_dim=d).to(DEVICE)

    for t in range(cfg["rounds"]):
        participants = [k for k in range(K) if rng.uniform() > cfg["dropout_q"]]
        if len(participants) < cfg["participation_min"]:
            participants = list(rng.choice(K, size=cfg["participation_min"], replace=False))

        m_count = int(np.floor(cfg["malicious_frac"] * len(participants)))
        malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()

        deltas, weights = [], []
        ref_sd = None

        for k in participants:
            local_model = train_one_client_fedprox(model, silo_data[k]["train_loader"], cfg, seed_offset=1000*t + k)
            delta_vec, ref_sd = compute_delta(model, local_model)
            protected = clip_and_noise(delta_vec, C=cfg["clip_C"], sigma=cfg["dp_sigma"], seed=cfg["seed"] + 1000*t + k)
            if k in malicious:
                protected = -cfg["poison_scale"] * protected
            deltas.append(protected)
            weights.append(silo_data[k]["n_train"])

        if aggregator == "fedavg":
            agg_delta = agg_fedavg(deltas, weights)
        else:
            agg_delta = agg_trimmed_mean(deltas, beta=TRIM_BETA)

        model = apply_delta_to_model(model, agg_delta, ref_sd, gamma=cfg["server_gamma"])

    metrics = eval_with_robust_thetas(model, default_theta=cfg["threshold"])
    return metrics

alpha_grid = [0.0, 0.1, 0.2, 0.3]
scale_grid = [1.0, 5.0, 10.0]

rows = []
for a in alpha_grid:
    for s in scale_grid:
        for agg in ["fedavg", "trimmean"]:
            m = run_poisoned(agg, malicious_frac=a, poison_scale=s)
            rows.append({
                "malicious_frac": a,
                "poison_scale": s,
                "aggregator": agg,
                "acc": float(m["acc"]),
                "precision": float(m["precision"]),
                "recall": float(m["recall"]),
                "f1": float(m["f1"]),
                "fpr": float(m["fpr"]),
                "tp": int(m["tp"]), "tn": int(m["tn"]), "fp": int(m["fp"]), "fn": int(m["fn"]),
            })
        print(f"✅ Done: alpha={a} scale={s}")

df = pd.DataFrame(rows).sort_values(["malicious_frac", "poison_scale", "aggregator"])
display(df)

# Save sweep results
out_path = Path("runs") / f"poison_sweep_{time.strftime('%Y%m%d_%H%M%S', time.gmtime())}.csv"
df.to_csv(out_path, index=False)
print("✅ Saved sweep table:", out_path)


**Next step — Step 14: Create a clean repo structure + write reusable modules + single entrypoint script**

In [ ]:
# Step 14 (FIXED): Generate clean repo code (src/, scripts/, configs/, README) safely

from pathlib import Path
import json

ROOT = Path(".").resolve()

# Create folders
for p in [
    ROOT / "src",
    ROOT / "src" / "data",
    ROOT / "src" / "models",
    ROOT / "src" / "fl",
    ROOT / "src" / "eval",
    ROOT / "scripts",
    ROOT / "configs",
]:
    p.mkdir(parents=True, exist_ok=True)

# Add __init__.py files
for p in [
    ROOT / "src" / "__init__.py",
    ROOT / "src" / "data" / "__init__.py",
    ROOT / "src" / "models" / "__init__.py",
    ROOT / "src" / "fl" / "__init__.py",
    ROOT / "src" / "eval" / "__init__.py",
]:
    p.write_text("")

# -------------------------
# src/models/mlp.py
# -------------------------
mlp_py = "\n".join([
"import torch",
"import torch.nn as nn",
"",
"class MLPIDS(nn.Module):",
"    \"\"\"Lightweight MLP IDS for tabular flow/session features. Outputs logits.\"\"\"",
"",
"    def __init__(self, input_dim: int, hidden=(256,128,64), dropout=0.30, batch_norm=False):",
"        super().__init__()",
"        layers = []",
"        in_dim = input_dim",
"        for h in hidden:",
"            layers.append(nn.Linear(in_dim, h))",
"            if batch_norm:",
"                layers.append(nn.BatchNorm1d(h))",
"            layers.append(nn.ReLU())",
"            layers.append(nn.Dropout(dropout))",
"            in_dim = h",
"        self.net = nn.Sequential(*layers)",
"        self.out = nn.Linear(in_dim, 1)",
"",
"    def forward(self, x):",
"        return self.out(self.net(x)).squeeze(-1)",
"",
])
(ROOT / "src/models/mlp.py").write_text(mlp_py)

# -------------------------
# src/data/preprocess_edgeiiotset.py
# -------------------------
preprocess_py = "\n".join([
"import json",
"from pathlib import Path",
"import numpy as np",
"import pandas as pd",
"",
"DROP_COLUMNS_DEFAULT = [",
"    \"frame.time\", \"ip.src_host\", \"ip.dst_host\", \"arp.src.proto_ipv4\", \"arp.dst.proto_ipv4\",",
"    \"http.file_data\", \"http.request.full_uri\", \"icmp.transmit_timestamp\",",
"    \"http.request.uri.query\", \"tcp.options\", \"tcp.payload\", \"tcp.srcport\",",
"    \"tcp.dstport\", \"udp.port\", \"mqtt.msg\"",
"]",
"",
"PREFERRED_CAT_COLS = [",
"    \"http.request.method\", \"http.referer\", \"http.request.version\", \"dns.qry.name.len\",",
"    \"mqtt.conack.flags\", \"mqtt.protoname\", \"mqtt.topic\"",
"]",
"",
"def preprocess_edgeiiotset(csv_path: str, out_npz: str, out_meta: str, drop_columns=None):",
"    csv_path = Path(csv_path)",
"    out_npz = Path(out_npz); out_npz.parent.mkdir(parents=True, exist_ok=True)",
"    out_meta = Path(out_meta); out_meta.parent.mkdir(parents=True, exist_ok=True)",
"",
"    df = pd.read_csv(csv_path, low_memory=False)",
"",
"    drop_columns = DROP_COLUMNS_DEFAULT if drop_columns is None else drop_columns",
"    existing = [c for c in drop_columns if c in df.columns]",
"    df.drop(existing, axis=1, inplace=True)",
"",
"    df.dropna(axis=0, how=\"any\", inplace=True)",
"    df.drop_duplicates(keep=\"first\", inplace=True)",
"    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)",
"",
"    if \"Attack_type\" not in df.columns:",
"        raise ValueError(\"Expected 'Attack_type' column in Edge-IIoTset CSV.\")",
"",
"    attack_type = df[\"Attack_type\"].astype(str).to_numpy()",
"    y = (attack_type != \"Normal\").astype(np.int64)",
"",
"    feature_df = df.drop(columns=[\"Attack_type\"])",
"",
"    preferred_present = [c for c in PREFERRED_CAT_COLS if c in feature_df.columns]",
"    object_cols = [c for c in feature_df.columns if feature_df[c].dtype == \"object\"]",
"    if preferred_present:",
"        cat_cols = sorted(set(preferred_present + object_cols))",
"    else:",
"        cat_cols = object_cols",
"",
"    X_df = pd.get_dummies(feature_df, columns=cat_cols, drop_first=False)",
"    if X_df.isna().any().any():",
"        raise ValueError(\"NaNs found after encoding; check preprocessing pipeline.\")",
"",
"    X = X_df.to_numpy(dtype=np.float32)",
"    np.savez_compressed(out_npz, X=X, y=y, attack_type=attack_type)",
"",
"    meta = {",
"        \"csv_path\": str(csv_path),",
"        \"n_rows\": int(X.shape[0]),",
"        \"n_features\": int(X.shape[1]),",
"        \"feature_names\": list(X_df.columns),",
"        \"encoded_categorical_columns\": cat_cols,",
"        \"label_definition\": \"Label=1 if Attack_type!='Normal' else 0\",",
"        \"dropped_columns\": existing,",
"    }",
"    out_meta.write_text(json.dumps(meta, indent=2))",
"    return str(out_npz), str(out_meta)",
"",
])
(ROOT / "src/data/preprocess_edgeiiotset.py").write_text(preprocess_py)

# -------------------------
# src/data/splits.py
# -------------------------
splits_py = "\n".join([
"import json",
"import numpy as np",
"from pathlib import Path",
"from sklearn.model_selection import train_test_split",
"",
"def dirichlet_partition_indices(y, K, alpha, seed=42, min_per_silo=10, max_tries=500):",
"    rng = np.random.default_rng(seed)",
"    classes = np.unique(y)",
"    best_silos = None",
"    best_min_size = -1",
"    for _ in range(max_tries):",
"        silos = [[] for _ in range(K)]",
"        for c in classes:",
"            idx_c = np.where(y == c)[0]",
"            rng.shuffle(idx_c)",
"            props = rng.dirichlet(alpha=np.full(K, alpha))",
"            counts = rng.multinomial(len(idx_c), props)",
"            start = 0",
"            for k in range(K):",
"                cnt = counts[k]",
"                if cnt > 0:",
"                    silos[k].extend(idx_c[start:start+cnt].tolist())",
"                start += cnt",
"        sizes = np.array([len(s) for s in silos], dtype=int)",
"        min_size = int(sizes.min())",
"        if min_size > best_min_size:",
"            best_min_size = min_size",
"            best_silos = [np.array(s, dtype=np.int64) for s in silos]",
"        if min_size >= min_per_silo:",
"            for k in range(K):",
"                rng.shuffle(silos[k])",
"            return [np.array(s, dtype=np.int64) for s in silos], True, best_min_size",
"    return best_silos, False, best_min_size",
"",
"def split_silo_indices(idx, y, seed=42, ratios=(0.70,0.15,0.15)):",
"    idx = np.array(idx, dtype=np.int64)",
"    y_s = y[idx]",
"    if len(idx) < 30 or len(np.unique(y_s)) < 2:",
"        n = len(idx)",
"        n_train = max(1, int(ratios[0]*n))",
"        n_val = max(0, int(ratios[1]*n))",
"        perm = np.random.default_rng(seed).permutation(idx)",
"        tr = perm[:n_train]",
"        va = perm[n_train:n_train+n_val]",
"        te = perm[n_train+n_val:]",
"        return tr, va, te",
"    test_val_frac = ratios[1] + ratios[2]",
"    strat_ok = (len(np.unique(y_s)) > 1) and (np.min(np.bincount(y_s)) >= 2)",
"    tr, temp = train_test_split(idx, test_size=test_val_frac, random_state=seed, shuffle=True,",
"                                stratify=y_s if strat_ok else None)",
"    y_temp = y[temp]",
"    val_frac_of_temp = ratios[1] / (ratios[1] + ratios[2])",
"    strat_ok2 = (len(np.unique(y_temp)) > 1) and (np.min(np.bincount(y_temp)) >= 2)",
"    va, te = train_test_split(temp, test_size=(1.0 - val_frac_of_temp), random_state=seed, shuffle=True,",
"                              stratify=y_temp if strat_ok2 else None)",
"    return tr, va, te",
"",
"def make_splits_and_save(encoded_npz, out_npz, out_summary_json, K=10, alpha=0.3, seed=42,",
"                         ratios=(0.70,0.15,0.15), min_per_silo=10):",
"    data = np.load(encoded_npz, allow_pickle=True)",
"    y = data[\"y\"].astype(int)",
"    silos, met, best_min = dirichlet_partition_indices(y, K=K, alpha=alpha, seed=seed, min_per_silo=min_per_silo)",
"    splits = {\"train\": [], \"val\": [], \"test\": []}",
"    stats = []",
"    for k in range(K):",
"        tr, va, te = split_silo_indices(silos[k], y, seed=seed, ratios=ratios)",
"        splits[\"train\"].append(tr); splits[\"val\"].append(va); splits[\"test\"].append(te)",
"        yk = y[silos[k]]",
"        stats.append({",
"            \"silo\": k,",
"            \"n_total\": int(len(silos[k])),",
"            \"n0\": int((yk==0).sum()),",
"            \"n1\": int((yk==1).sum()),",
"            \"intrusion_rate\": float((yk==1).mean()) if len(yk) else 0.0,",
"            \"n_train\": int(len(tr)),",
"            \"n_val\": int(len(va)),",
"            \"n_test\": int(len(te)),",
"        })",
"    out_npz = Path(out_npz); out_npz.parent.mkdir(parents=True, exist_ok=True)",
"    np.savez_compressed(out_npz,",
"        train=np.array(splits[\"train\"], dtype=object),",
"        val=np.array(splits[\"val\"], dtype=object),",
"        test=np.array(splits[\"test\"], dtype=object),",
"        alpha=np.array([alpha], dtype=np.float32),",
"        K=np.array([K], dtype=np.int32),",
"        seed=np.array([seed], dtype=np.int32),",
"    )",
"    out_summary_json = Path(out_summary_json); out_summary_json.parent.mkdir(parents=True, exist_ok=True)",
"    out_summary_json.write_text(json.dumps({",
"        \"alpha\": alpha, \"K\": K, \"seed\": seed, \"ratios\": ratios,",
"        \"min_per_silo_target\": min_per_silo,",
"        \"met_min_per_silo_target\": bool(met),",
"        \"best_min_silo_size_seen\": int(best_min),",
"        \"silo_stats\": stats",
"    }, indent=2))",
"    return str(out_npz), str(out_summary_json)",
"",
])
(ROOT / "src/data/splits.py").write_text(splits_py)

# -------------------------
# src/data/loaders.py
# -------------------------
loaders_py = "\n".join([
"import numpy as np",
"import torch",
"from torch.utils.data import Dataset, DataLoader",
"",
"class TabularDataset(Dataset):",
"    def __init__(self, X, y):",
"        self.X = torch.from_numpy(X).float()",
"        self.y = torch.from_numpy(y).long()",
"    def __len__(self): return self.X.shape[0]",
"    def __getitem__(self, i): return self.X[i], self.y[i]",
"",
"def fit_local_norm(X_train):",
"    mu = X_train.mean(axis=0, dtype=np.float64)",
"    sigma = X_train.std(axis=0, dtype=np.float64)",
"    return mu.astype(np.float32), sigma.astype(np.float32)",
"",
"def apply_local_norm(X, mu, sigma, eps=1e-8):",
"    return (X - mu) / (sigma + eps)",
"",
"def build_silo_loaders(encoded_npz, splits_npz, batch_size=256, eps=1e-8):",
"    data = np.load(encoded_npz, allow_pickle=True)",
"    X = data[\"X\"].astype(np.float32)",
"    y = data[\"y\"].astype(np.int64)",
"",
"    spl = np.load(splits_npz, allow_pickle=True)",
"    train_idxs = spl[\"train\"]; val_idxs = spl[\"val\"]; test_idxs = spl[\"test\"]",
"    K = int(spl[\"K\"][0])",
"",
"    silo = []",
"    for k in range(K):",
"        tr = np.array(train_idxs[k], dtype=np.int64)",
"        va = np.array(val_idxs[k], dtype=np.int64)",
"        te = np.array(test_idxs[k], dtype=np.int64)",
"",
"        X_tr, y_tr = X[tr], y[tr]",
"        X_va, y_va = X[va], y[va]",
"        X_te, y_te = X[te], y[te]",
"",
"        mu_k, sigma_k = fit_local_norm(X_tr)",
"        X_tr = apply_local_norm(X_tr, mu_k, sigma_k, eps)",
"        X_va = apply_local_norm(X_va, mu_k, sigma_k, eps)",
"        X_te = apply_local_norm(X_te, mu_k, sigma_k, eps)",
"",
"        silo.append({",
"            \"train_loader\": DataLoader(TabularDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True),",
"            \"val_loader\":   DataLoader(TabularDataset(X_va, y_va), batch_size=batch_size, shuffle=False),",
"            \"test_loader\":  DataLoader(TabularDataset(X_te, y_te), batch_size=batch_size, shuffle=False),",
"            \"n_train\": int(len(tr)), \"n_val\": int(len(va)), \"n_test\": int(len(te)),",
"        })",
"    return silo, X.shape[1], K",
"",
])
(ROOT / "src/data/loaders.py").write_text(loaders_py)

# -------------------------
# src/fl/aggregators.py
# -------------------------
aggs_py = "\n".join([
"import numpy as np",
"import torch",
"",
"def agg_fedavg(deltas, weights):",
"    W = float(np.sum(weights))",
"    out = torch.zeros_like(deltas[0])",
"    for d, w in zip(deltas, weights):",
"        out += d * (float(w) / W)",
"    return out",
"",
"def agg_trimmed_mean(deltas, beta=0.10):",
"    m = len(deltas)",
"    X = torch.stack(deltas, dim=0)",
"    trim = int(np.floor(beta * m))",
"    if 2 * trim >= m:",
"        return torch.mean(X, dim=0)",
"    Xs, _ = torch.sort(X, dim=0)",
"    Xk = Xs[trim:m-trim, :]",
"    return torch.mean(Xk, dim=0)",
"",
])
(ROOT / "src/fl/aggregators.py").write_text(aggs_py)

# -------------------------
# src/fl/attacks.py
# -------------------------
(ROOT / "src/fl/attacks.py").write_text("def signflip_poison(delta_vec, scale=5.0):\n    return -scale * delta_vec\n")

# -------------------------
# src/fl/client.py
# -------------------------
client_py = "\n".join([
"import copy",
"import torch",
"",
"def get_param_vector(model):",
"    return torch.cat([p.view(-1) for p in model.parameters()])",
"",
"def fedprox_penalty(model, w0_vec):",
"    w_vec = get_param_vector(model)",
"    return 0.5 * torch.sum((w_vec - w0_vec) ** 2)",
"",
"def train_one_client_fedprox(global_model, train_loader, criterion, device, cfg, seed_offset=0):",
"    torch.manual_seed(cfg[\"seed\"] + seed_offset)",
"    model = copy.deepcopy(global_model).to(device)",
"    opt = torch.optim.Adam(model.parameters(), lr=cfg[\"lr\"], weight_decay=cfg[\"weight_decay\"])",
"    with torch.no_grad():",
"        w0 = get_param_vector(model).detach()",
"    model.train()",
"    for _ in range(cfg[\"local_epochs\"]):",
"        for xb, yb in train_loader:",
"            xb = xb.to(device)",
"            yb = yb.float().to(device)",
"            opt.zero_grad()",
"            logits = model(xb)",
"            base = criterion(logits, yb)",
"            prox = cfg[\"fedprox_mu\"] * fedprox_penalty(model, w0)",
"            loss = base + prox",
"            loss.backward()",
"            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg[\"grad_clip\"])",
"            opt.step()",
"    return model",
"",
])
(ROOT / "src/fl/client.py").write_text(client_py)

# -------------------------
# src/fl/utils.py
# -------------------------
utils_py = "\n".join([
"import torch",
"",
"def state_dict_to_vector(sd):",
"    return torch.cat([v.detach().reshape(-1).float().cpu() for v in sd.values() if torch.is_tensor(v)])",
"",
"def vector_to_state_dict_like(vec, ref_sd):",
"    out, off = {}, 0",
"    for k, v in ref_sd.items():",
"        if torch.is_tensor(v):",
"            n = v.numel()",
"            out[k] = vec[off:off+n].view_as(v).to(v.dtype)",
"            off += n",
"        else:",
"            out[k] = v",
"    return out",
"",
"def compute_delta(global_model, local_model):",
"    g_sd = global_model.state_dict()",
"    l_sd = local_model.state_dict()",
"    return state_dict_to_vector(l_sd) - state_dict_to_vector(g_sd), g_sd",
"",
"def clip_and_noise(delta_vec, C=1.0, sigma=0.0, seed=42):",
"    delta = delta_vec.clone()",
"    norm = torch.linalg.norm(delta, ord=2)",
"    scale = max(1.0, float(norm.item()) / float(C))",
"    delta = delta / scale",
"    if sigma and sigma > 0.0:",
"        gen = torch.Generator(device=\"cpu\").manual_seed(int(seed))",
"        delta = delta + torch.normal(0.0, float(sigma), size=delta.shape, generator=gen)",
"    return delta",
"",
"def apply_delta_to_model(global_model, delta_vec, ref_sd, gamma=1.0):",
"    g_sd = global_model.state_dict()",
"    delta_sd = vector_to_state_dict_like(delta_vec, ref_sd)",
"    new_sd = {}",
"    for k, v in g_sd.items():",
"        if torch.is_tensor(v):",
"            new_sd[k] = v + gamma * delta_sd[k].to(v.device)",
"        else:",
"            new_sd[k] = v",
"    global_model.load_state_dict(new_sd)",
"    return global_model",
"",
])
(ROOT / "src/fl/utils.py").write_text(utils_py)

# -------------------------
# src/eval/thresholds.py
# -------------------------
thresholds_py = "\n".join([
"import numpy as np",
"import torch",
"",
"def metrics_at_threshold(probs, y_true, thr):",
"    y_pred = (probs >= thr).astype(int)",
"    tp = int(((y_pred==1)&(y_true==1)).sum())",
"    tn = int(((y_pred==0)&(y_true==0)).sum())",
"    fp = int(((y_pred==1)&(y_true==0)).sum())",
"    fn = int(((y_pred==0)&(y_true==1)).sum())",
"    precision = tp / (tp + fp + 1e-12)",
"    recall    = tp / (tp + fn + 1e-12)",
"    f1        = 2*precision*recall / (precision + recall + 1e-12)",
"    acc       = (tp + tn) / max(1, tp+tn+fp+fn)",
"    fpr       = fp / (fp + tn + 1e-12)",
"    return {\"acc\":acc,\"precision\":precision,\"recall\":recall,\"f1\":f1,\"fpr\":fpr,",
"            \"tp\":tp,\"tn\":tn,\"fp\":fp,\"fn\":fn,\"pred_pos_rate\":float(y_pred.mean())}",
"",
"@torch.no_grad()",
"def collect_probs(model, loader, device):",
"    probs_list, y_list = [], []",
"    for xb, yb in loader:",
"        xb = xb.to(device)",
"        probs_list.append(torch.sigmoid(model(xb)).cpu().numpy())",
"        y_list.append(yb.cpu().numpy())",
"    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)",
"",
"def robust_theta_from_fpr(probs_val, y_val, default_theta, target_fpr=0.05, theta_min=0.05, theta_max=0.95, min_benign_val=200):",
"    benign = probs_val[y_val == 0]",
"    if benign.size < min_benign_val:",
"        return default_theta, f\"fallback_default (benign_val={benign.size})\"",
"    thr = float(np.quantile(benign, 1.0 - target_fpr))",
"    thr = float(np.clip(thr, theta_min, theta_max))",
"    return thr, \"quantile_benign\"",
"",
])
(ROOT / "src/eval/thresholds.py").write_text(thresholds_py)

# -------------------------
# scripts/run_experiment.py
# -------------------------
run_script = "\n".join([
"import argparse, json, time",
"from pathlib import Path",
"import numpy as np",
"import torch",
"import torch.nn as nn",
"",
"from src.data.loaders import build_silo_loaders",
"from src.models.mlp import MLPIDS",
"from src.fl.client import train_one_client_fedprox",
"from src.fl.aggregators import agg_fedavg, agg_trimmed_mean",
"from src.fl.attacks import signflip_poison",
"from src.fl.utils import compute_delta, clip_and_noise, apply_delta_to_model",
"from src.eval.thresholds import collect_probs, metrics_at_threshold, robust_theta_from_fpr",
"",
"def eval_with_robust_thetas(model, silo_data, device, default_theta, calib):",
"    K = len(silo_data)",
"    per_silo = []",
"    for k in range(K):",
"        probs_val, y_val = collect_probs(model, silo_data[k]['val_loader'], device)",
"        theta_k, _ = robust_theta_from_fpr(",
"            probs_val, y_val, default_theta,",
"            target_fpr=calib['target_fpr'], theta_min=calib['theta_min'], theta_max=calib['theta_max'],",
"            min_benign_val=calib['min_benign_val']",
"        )",
"        m_val = metrics_at_threshold(probs_val, y_val, theta_k)",
"        if m_val['pred_pos_rate'] > calib['max_pred_pos']:",
"            theta_k = min(calib['theta_max'], max(theta_k, 0.80))",
"        elif m_val['pred_pos_rate'] < calib['min_pred_pos']:",
"            theta_k = max(calib['theta_min'], min(theta_k, 0.20))",
"",
"        probs_test, y_test = collect_probs(model, silo_data[k]['test_loader'], device)",
"        per_silo.append(metrics_at_threshold(probs_test, y_test, theta_k))",
"",
"    tp = sum(m['tp'] for m in per_silo); tn = sum(m['tn'] for m in per_silo)",
"    fp = sum(m['fp'] for m in per_silo); fn = sum(m['fn'] for m in per_silo)",
"    precision = tp / (tp + fp + 1e-12)",
"    recall    = tp / (tp + fn + 1e-12)",
"    f1        = 2*precision*recall / (precision + recall + 1e-12)",
"    acc       = (tp + tn) / max(1, tp+tn+fp+fn)",
"    fpr       = fp / (fp + tn + 1e-12)",
"    return {'acc':float(acc),'precision':float(precision),'recall':float(recall),'f1':float(f1),",
"            'fpr':float(fpr),'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn)}",
"",
"def main():",
"    ap = argparse.ArgumentParser()",
"    ap.add_argument('--config', type=str, required=True)",
"    args = ap.parse_args()",
"",
"    cfg = json.loads(Path(args.config).read_text())",
"    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')",
"",
"    run_id = time.strftime('run_repo_%Y%m%d_%H%M%S', time.gmtime())",
"    run_dir = Path('runs') / run_id",
"    run_dir.mkdir(parents=True, exist_ok=True)",
"    (run_dir / 'config.json').write_text(json.dumps(cfg, indent=2))",
"",
"    silo_data, d, K = build_silo_loaders(cfg['encoded_npz'], cfg['splits_npz'], batch_size=cfg['batch_size'])",
"",
"    model = MLPIDS(d, hidden=tuple(cfg['hidden']), dropout=cfg['dropout'], batch_norm=cfg['batch_norm']).to(device)",
"    pos_weight = torch.tensor([cfg['pos_weight']], device=device)",
"    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)",
"",
"    rng = np.random.default_rng(cfg['seed'])",
"",
"    for t in range(cfg['rounds']):",
"        participants = [k for k in range(K) if rng.uniform() > cfg['dropout_q']]",
"        if len(participants) < cfg['participation_min']:",
"            participants = list(rng.choice(K, size=cfg['participation_min'], replace=False))",
"",
"        m_count = int(np.floor(cfg['malicious_frac'] * len(participants)))",
"        malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()",
"",
"        deltas, weights = [], []",
"        ref_sd = None",
"        for k in participants:",
"            local_model = train_one_client_fedprox(model, silo_data[k]['train_loader'], criterion, device, cfg, seed_offset=1000*t + k)",
"            delta_vec, ref_sd = compute_delta(model, local_model)",
"            protected = clip_and_noise(delta_vec, C=cfg['clip_C'], sigma=cfg['dp_sigma'], seed=cfg['seed'] + 1000*t + k)",
"            if k in malicious:",
"                protected = signflip_poison(protected, scale=cfg['poison_scale'])",
"            deltas.append(protected)",
"            weights.append(silo_data[k]['n_train'])",
"",
"        if cfg['aggregator'] == 'fedavg':",
"            agg_delta = agg_fedavg(deltas, weights)",
"        else:",
"            agg_delta = agg_trimmed_mean(deltas, beta=cfg['trim_beta'])",
"",
"        model = apply_delta_to_model(model, agg_delta, ref_sd, gamma=cfg['server_gamma'])",
"        print(f\"{cfg['aggregator']} round {t} | participants={len(participants)} malicious={len(malicious)}\")",
"",
"    metrics = eval_with_robust_thetas(model, silo_data, device, cfg['threshold'], cfg['calibration'])",
"    (run_dir / 'final_global_test.json').write_text(json.dumps(metrics, indent=2))",
"    torch.save(model.state_dict(), run_dir / 'model_final.pt')",
"    print('✅ Final metrics:', metrics)",
"    print('✅ Saved run:', run_dir)",
"",
"if __name__ == '__main__':",
"    main()",
"",
])
(ROOT / "scripts/run_experiment.py").write_text(run_script)

# -------------------------
# configs/base.json
# -------------------------
base_cfg = {
    "seed": 42,
    "encoded_npz": "data/processed/edgeiiotset_encoded.npz",
    "splits_npz": "data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz",
    "batch_size": 256,

    "hidden": [256, 128, 64],
    "dropout": 0.30,
    "batch_norm": False,
    "pos_weight": 2.0,

    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 6,

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,

    "clip_C": 1.0,
    "dp_sigma": 0.0,

    "threshold": 0.5,

    "aggregator": "fedavg",      # "fedavg" or "trimmean"
    "trim_beta": 0.10,

    "malicious_frac": 0.2,
    "poison_scale": 5.0,

    "calibration": {
        "target_fpr": 0.05,
        "theta_min": 0.05,
        "theta_max": 0.95,
        "min_benign_val": 200,
        "max_pred_pos": 0.90,
        "min_pred_pos": 0.01
    }
}
(ROOT / "configs/base.json").write_text(json.dumps(base_cfg, indent=2))

# -------------------------
# README.md
# -------------------------
readme = "\n".join([
"# Cross-silo FL IDS (Edge-IIoTset)",
"",
"Minimal, reproducible code for:",
"- preprocessing Edge-IIoTset CSV into encoded NPZ",
"- Dirichlet non-IID partitioning (K=10)",
"- FedProx local training + FedAvg / TrimMean aggregation",
"- sign-flip poisoning robustness evaluation",
"- robust per-silo threshold calibration (FPR-target)",
"",
"## Run",
"",
"After you have generated `data/processed/edgeiiotset_encoded.npz` and a splits file:",
"",
"```bash",
"python scripts/run_experiment.py --config configs/base.json",
"```",
"",
"Artifacts are saved under `runs/run_repo_*`.",
"",
])
(ROOT / "README.md").write_text(readme)

print("✅ Repo files created successfully (fixed Step 14).")
print("Check: src/, scripts/, configs/, README.md")


**Step 15 — Add ready-to-run configs + a sweep runner that aggregates results into one CSV**

In [ ]:
# Step 15: Create experiment configs + sweep runner script (aggregates to one CSV)

from pathlib import Path
import json

ROOT = Path(".").resolve()
CFG_DIR = ROOT / "configs"
SCR_DIR = ROOT / "scripts"
CFG_DIR.mkdir(parents=True, exist_ok=True)
SCR_DIR.mkdir(parents=True, exist_ok=True)

base = json.loads((CFG_DIR / "base.json").read_text())

def write_cfg(name, updates):
    cfg = dict(base)
    # deep update for nested calibration if present
    if "calibration" in updates:
        cal = dict(cfg["calibration"])
        cal.update(updates["calibration"])
        updates = dict(updates)
        updates["calibration"] = cal
    cfg.update(updates)
    (CFG_DIR / name).write_text(json.dumps(cfg, indent=2))

# Clean baselines
write_cfg("clean_fedavg.json", {
    "aggregator": "fedavg",
    "malicious_frac": 0.0,
    "poison_scale": 1.0,
})
write_cfg("clean_trimmean.json", {
    "aggregator": "trimmean",
    "trim_beta": 0.10,
    "malicious_frac": 0.0,
    "poison_scale": 1.0,
})

# Example attacked configs
write_cfg("poison_alpha20_scale5_fedavg.json", {
    "aggregator": "fedavg",
    "malicious_frac": 0.2,
    "poison_scale": 5.0,
})
write_cfg("poison_alpha20_scale5_trimmean.json", {
    "aggregator": "trimmean",
    "trim_beta": 0.10,
    "malicious_frac": 0.2,
    "poison_scale": 5.0,
})

# Sweep spec (grid)
sweep = {
    "alpha_grid": [0.0, 0.1, 0.2, 0.3],
    "scale_grid": [1.0, 5.0, 10.0],
    "aggregators": ["fedavg", "trimmean"],
    "trim_beta": 0.10,
    "base_config": "configs/base.json",
    "out_csv": "runs/sweep_repo_results.csv"
}
(CFG_DIR / "sweep_poison.json").write_text(json.dumps(sweep, indent=2))

# scripts/run_sweep.py
run_sweep = "\n".join([
"import argparse, json, subprocess, sys",
"from pathlib import Path",
"import pandas as pd",
"",
"def main():",
"    ap = argparse.ArgumentParser()",
"    ap.add_argument('--sweep', type=str, required=True, help='configs/sweep_poison.json')",
"    args = ap.parse_args()",
"",
"    sweep = json.loads(Path(args.sweep).read_text())",
"    base = json.loads(Path(sweep['base_config']).read_text())",
"    out_csv = Path(sweep['out_csv'])",
"    out_csv.parent.mkdir(parents=True, exist_ok=True)",
"",
"    rows = []",
"    tmp_cfg_dir = Path('configs/_generated')",
"    tmp_cfg_dir.mkdir(parents=True, exist_ok=True)",
"",
"    for a in sweep['alpha_grid']:",
"        for s in sweep['scale_grid']:",
"            for agg in sweep['aggregators']:",
"                cfg = dict(base)",
"                cfg['malicious_frac'] = float(a)",
"                cfg['poison_scale'] = float(s)",
"                cfg['aggregator'] = agg",
"                if agg == 'trimmean':",
"                    cfg['trim_beta'] = float(sweep.get('trim_beta', 0.10))",
"",
"                cfg_path = tmp_cfg_dir / f'cfg_a{a}_s{s}_{agg}.json'",
"                cfg_path.write_text(json.dumps(cfg, indent=2))",
"",
"                # Run experiment",
"                cmd = [sys.executable, 'scripts/run_experiment.py', '--config', str(cfg_path)]",
"                print('RUN:', ' '.join(cmd))",
"                subprocess.run(cmd, check=True)",
"",
"    # Collect results from runs/run_repo_*",
"    run_dirs = sorted(Path('runs').glob('run_repo_*'))",
"    for rd in run_dirs:",
"        cfg_path = rd / 'config.json'",
"        res_path = rd / 'final_global_test.json'",
"        if not (cfg_path.exists() and res_path.exists()):",
"            continue",
"        cfg = json.loads(cfg_path.read_text())",
"        res = json.loads(res_path.read_text())",
"        rows.append({",
"            'run_dir': str(rd),",
"            'aggregator': cfg.get('aggregator'),",
"            'trim_beta': cfg.get('trim_beta', None),",
"            'malicious_frac': cfg.get('malicious_frac'),",
"            'poison_scale': cfg.get('poison_scale'),",
"            'acc': res.get('acc'),",
"            'precision': res.get('precision'),",
"            'recall': res.get('recall'),",
"            'f1': res.get('f1'),",
"            'fpr': res.get('fpr'),",
"            'tp': res.get('tp'),",
"            'tn': res.get('tn'),",
"            'fp': res.get('fp'),",
"            'fn': res.get('fn'),",
"        })",
"",
"    df = pd.DataFrame(rows)",
"    df.to_csv(out_csv, index=False)",
"    print('✅ Wrote:', out_csv, '| rows:', len(df))",
"",
"if __name__ == '__main__':",
"    main()",
""
])
(SCR_DIR / "run_sweep.py").write_text(run_sweep)

print("✅ Created configs:")
print(" - configs/clean_fedavg.json")
print(" - configs/clean_trimmean.json")
print(" - configs/poison_alpha20_scale5_fedavg.json")
print(" - configs/poison_alpha20_scale5_trimmean.json")
print(" - configs/sweep_poison.json")
print("✅ Created sweep runner: scripts/run_sweep.py")


**Step 16 — Add a dataset-from-Drive unzip + CSV discovery script**

In [ ]:
# Step 16: Create a script to unzip your private Drive dataset locally and locate the CSV

from pathlib import Path
import json

ROOT = Path(".").resolve()
(ROOT / "scripts").mkdir(parents=True, exist_ok=True)
(ROOT / "configs").mkdir(parents=True, exist_ok=True)
(ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)
(ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)

prepare_py = "\n".join([
"import argparse, zipfile, os",
"from pathlib import Path",
"",
"def find_csvs(root: Path):",
"    return sorted([p for p in root.rglob('*.csv') if p.is_file()])",
"",
"def main():",
"    ap = argparse.ArgumentParser()",
"    ap.add_argument('--zip_path', type=str, required=True, help='Path to archive.zip (Drive-mounted)')",
"    ap.add_argument('--out_dir', type=str, default='data/raw/edgeiiotset', help='Where to unzip')",
"    ap.add_argument('--pick_contains', type=str, default='DNN-EdgeIIoT-dataset.csv',",
"                    help='Preferred CSV filename substring to auto-select')",
"    args = ap.parse_args()",
"",
"    zip_path = Path(args.zip_path)",
"    out_dir = Path(args.out_dir)",
"    out_dir.mkdir(parents=True, exist_ok=True)",
"",
"    if not zip_path.exists():",
"        raise FileNotFoundError(f'Zip not found: {zip_path}')",
"",
"    # Unzip",
"    with zipfile.ZipFile(zip_path, 'r') as zf:",
"        zf.extractall(out_dir)",
"",
"    csvs = find_csvs(out_dir)",
"    if not csvs:",
"        raise RuntimeError(f'No CSV files found after unzip under: {out_dir}')",
"",
"    # Auto-pick",
"    chosen = None",
"    for p in csvs:",
"        if args.pick_contains in p.name:",
"            chosen = p",
"            break",
"    if chosen is None:",
"        # fallback: largest CSV",
"        chosen = max(csvs, key=lambda p: p.stat().st_size)",
"",
"    print('✅ Unzipped to:', out_dir)",
"    print('✅ Found CSVs:', len(csvs))",
"    print('✅ Selected CSV:', chosen)",
"    print(str(chosen))",
"",
"if __name__ == '__main__':",
"    main()",
""
])
(Path("scripts/prepare_data_from_zip.py")).write_text(prepare_py)

cfg = {
  "zip_path": "/content/drive/MyDrive/archive.zip",
  "out_dir": "data/raw/edgeiiotset",
  "pick_contains": "DNN-EdgeIIoT-dataset.csv"
}
(Path("configs/data_from_zip.json")).write_text(json.dumps(cfg, indent=2))

print("✅ Created:")
print(" - scripts/prepare_data_from_zip.py")
print(" - configs/data_from_zip.json")


**Step 17 — Preprocess the unzipped CSV → build encoded NPZ → create Dirichlet splits**

In [ ]:
# Step 17: Auto-find CSV in unzipped folder -> preprocess -> create Dirichlet splits

from pathlib import Path
import json

from src.data.preprocess_edgeiiotset import preprocess_edgeiiotset
from src.data.splits import make_splits_and_save

# Load the zip/unzip config from Step 16
cfg_zip = json.loads(Path("configs/data_from_zip.json").read_text())
RAW_DIR = Path(cfg_zip["out_dir"])
PICK = cfg_zip.get("pick_contains", "DNN-EdgeIIoT-dataset.csv")

assert RAW_DIR.exists(), f"Raw unzip folder not found: {RAW_DIR}. Run Step 16 unzip first."

# Find CSVs
csvs = sorted([p for p in RAW_DIR.rglob("*.csv") if p.is_file()])
assert len(csvs) > 0, f"No CSV files found under {RAW_DIR}"

# Pick preferred (by name substring) else largest
chosen = None
for p in csvs:
    if PICK in p.name:
        chosen = p
        break
if chosen is None:
    chosen = max(csvs, key=lambda p: p.stat().st_size)

print("✅ Found CSVs:", len(csvs))
print("✅ Selected CSV:", chosen)

# Output paths
ENC_NPZ  = Path("data/processed/edgeiiotset_encoded.npz")
ENC_META = Path("data/processed/edgeiiotset_encoded_meta.json")

SPLITS_NPZ = Path("data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz")
SPLITS_SUM = Path("data/processed/splits_edgeiiotset_dirichlet_alpha_0.3_summary.json")

# 1) Preprocess (CSV -> encoded NPZ)
preprocess_edgeiiotset(
    csv_path=str(chosen),
    out_npz=str(ENC_NPZ),
    out_meta=str(ENC_META),
)
print("✅ Wrote:", ENC_NPZ)
print("✅ Wrote:", ENC_META)

# 2) Create non-IID Dirichlet partition + per-silo 70/15/15 splits
make_splits_and_save(
    encoded_npz=str(ENC_NPZ),
    out_npz=str(SPLITS_NPZ),
    out_summary_json=str(SPLITS_SUM),
    K=10,
    alpha=0.3,
    seed=42,
    ratios=(0.70, 0.15, 0.15),
    min_per_silo=10,
)
print("✅ Wrote:", SPLITS_NPZ)
print("✅ Wrote:", SPLITS_SUM)

# Quick sanity: show where to point configs/base.json
print("\n📌 Update your configs to use:")
print("encoded_npz =", str(ENC_NPZ))
print("splits_npz  =", str(SPLITS_NPZ))


**Step 18 — Run the repo entrypoint (clean FedAvg + clean TrimMean) and verify artifacts**

In [ ]:
!PYTHONPATH=. python scripts/run_experiment.py --config configs/clean_fedavg.json


In [ ]:
!PYTHONPATH=. python scripts/run_experiment.py --config configs/clean_trimmean.json

**Verification cell**

In [ ]:
from pathlib import Path
import json

runs = sorted(Path("runs").glob("run_repo_*"))
print("✅ Found run_repo_*:", len(runs))
for rd in runs[-2:]:
    res = json.loads((rd / "final_global_test.json").read_text())
    cfg = json.loads((rd / "config.json").read_text())
    print("\nRun:", rd)
    print(" aggregator:", cfg.get("aggregator"), "malicious_frac:", cfg.get("malicious_frac"), "poison_scale:", cfg.get("poison_scale"))
    print(" metrics:", {k: res[k] for k in ["acc","precision","recall","f1","fpr"]})
